In [ ]:
# ============================================================
# CÉLULA 1 - IMPORTS E VERIFICAÇÃO DOS ARQUIVOS NA PASTA
# ============================================================

import pandas as pd
import re
import unicodedata
import os
from pathlib import Path
from difflib import SequenceMatcher
from itertools import combinations
from google.colab import files

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

# ============================================================
# 1. CONFIGURAÇÃO DA PASTA DOS ARQUIVOS
# ============================================================

# Se os arquivos estiverem na pasta principal do Colab, deixe "."
PASTA_ARQUIVOS = Path(".")

# Se estiverem em outra pasta, exemplo:
# PASTA_ARQUIVOS = Path("/content/dados")

# ============================================================
# 2. CONFIGURAÇÃO DOS ARQUIVOS DA PIPELINE
# ============================================================

ARQUIVO_ACOLHIMENTO_1 = "acolhimento.xlsx"
ARQUIVO_ACOLHIMENTO_2 = "acolhimento2.xlsx"

CAMINHO_ACOLHIMENTO_1 = PASTA_ARQUIVOS / ARQUIVO_ACOLHIMENTO_1
CAMINHO_ACOLHIMENTO_2 = PASTA_ARQUIVOS / ARQUIVO_ACOLHIMENTO_2

# ============================================================
# 3. VALIDAR SE OS ARQUIVOS EXISTEM NA PASTA
# ============================================================

arquivos_obrigatorios = [
    CAMINHO_ACOLHIMENTO_1,
    CAMINHO_ACOLHIMENTO_2
]

for caminho in arquivos_obrigatorios:
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}\n"
            f"Arquivos disponíveis na pasta atual: {os.listdir(PASTA_ARQUIVOS)}"
        )

# ============================================================
# 4. LER AS PLANILHAS
# ============================================================

df_acolhimento_1 = pd.read_excel(CAMINHO_ACOLHIMENTO_1)
df_acolhimento_2 = pd.read_excel(CAMINHO_ACOLHIMENTO_2)

# ============================================================
# 5. CONFERÊNCIA INICIAL
# ============================================================

print("Arquivos carregados com sucesso!")

print("\nBase Acolhimento 1:")
print(f"Arquivo: {CAMINHO_ACOLHIMENTO_1}")
print(f"Linhas: {len(df_acolhimento_1)}")
print(f"Colunas: {len(df_acolhimento_1.columns)}")
print(df_acolhimento_1.columns.tolist())

print("\nBase Acolhimento 2:")
print(f"Arquivo: {CAMINHO_ACOLHIMENTO_2}")
print(f"Linhas: {len(df_acolhimento_2)}")
print(f"Colunas: {len(df_acolhimento_2.columns)}")
print(df_acolhimento_2.columns.tolist())

print("\nPrévia Acolhimento 1:")
display(df_acolhimento_1.head())

print("\nPrévia Acolhimento 2:")
display(df_acolhimento_2.head())

In [ ]:
# ============================================================
# CÉLULA 2 - CONFIGURAÇÕES DOS NOMES DOS ARQUIVOS
# ============================================================

# Arquivos de entrada
ARQUIVO_ACOLHIMENTO_1 = "acolhimento.xlsx"
ARQUIVO_ACOLHIMENTO_2 = "acolhimento2.xlsx"

# Não usaremos mais guarda
# Não usaremos adoção

# Nome do arquivo final exportado
ARQUIVO_SAIDA = "PLANILHA_FINAL_PIPELINE_TRATADA.xlsx"

# Nome das abas do Excel final
ABA_PLANILHA_FINAL = "Planilha Final"
ABA_AUDITORIA_IDADE = "Auditoria Idade"
ABA_AUDITORIA_NOMES = "Auditoria Nomes"
ABA_REPETIDOS_MEDIDA = "Repetidos Medida Protetiva"
ABA_RESUMO = "Resumo"

# Situação única da planilha final
SITUACAO_PADRAO = "Acolhido"

# Separador usado quando houver informações diferentes para a mesma criança
SEPARADOR_VALORES = " | "

# Colunas principais que devem aparecer no início da planilha final
COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

# Colunas que deverão ser removidas antes da exportação final, caso existam
COLUNAS_AUXILIARES_REMOVER = [
    "__chave__",
    "__nome_chave__",
    "__idade_chave__",
    "__origem__",
    "__linha_original__",
    "Origem Acolhimento",
    "Tipo",
    "Órgão Julgador",
]

# Arquivos obrigatórios da pipeline
ARQUIVOS_OBRIGATORIOS = [
    ARQUIVO_ACOLHIMENTO_1,
    ARQUIVO_ACOLHIMENTO_2
]

# Conferência das configurações
print("Configurações carregadas com sucesso!")
print(f"Acolhimento 1: {ARQUIVO_ACOLHIMENTO_1}")
print(f"Acolhimento 2: {ARQUIVO_ACOLHIMENTO_2}")
print("Guarda: não será usada")
print("Adoção: não será usada")
print(f"Situação padrão: {SITUACAO_PADRAO}")
print(f"Arquivo final: {ARQUIVO_SAIDA}")

In [ ]:
# ============================================================
# CÉLULA 3 - FUNÇÕES AUXILIARES GERAIS
# ============================================================

def normalizar_texto(txt):
    """
    Normaliza texto para comparação:
    - remove acentos
    - deixa minúsculo
    - remove espaços duplicados
    """

    if pd.isna(txt):
        return ""

    txt = str(txt).strip().lower()
    txt = unicodedata.normalize("NFKD", txt)
    txt = "".join(c for c in txt if not unicodedata.combining(c))
    txt = re.sub(r"[^a-z0-9\s]", " ", txt)
    txt = re.sub(r"\s+", " ", txt)

    return txt.strip()

def separar_orgao_julgador_generico(valor):
    """
    Divide Órgão Julgador em:
    - Município: parte antes do primeiro "-"
    - Vara/Unidade: parte depois do primeiro "-"

    Exemplo:
    'SALVADOR - 1ª VARA DA INFÂNCIA'
    vira:
    Município = Salvador
    Vara = 1ª VARA DA INFÂNCIA
    """

    texto = limpar_valor(valor)

    if texto == "":
        return pd.Series([pd.NA, pd.NA])

    partes = re.split(r"\s*\|\s*", texto)

    municipios = []
    varas = []

    for parte in partes:
        parte = limpar_valor(parte)

        if parte == "":
            continue

        split = re.split(r"\s*[-–—]\s*", parte, maxsplit=1)

        if len(split) == 1:
            municipio = split[0].strip()
            vara = ""
        else:
            municipio = split[0].strip()
            vara = split[1].strip()

        municipio = capitalizar_texto(municipio)

        if not valor_vazio(municipio):
            if normalizar_texto(municipio) not in [normalizar_texto(m) for m in municipios]:
                municipios.append(municipio)

        if not valor_vazio(vara):
            if normalizar_texto(vara) not in [normalizar_texto(v) for v in varas]:
                varas.append(vara)

    municipio_final = SEPARADOR_VALORES.join(municipios) if municipios else pd.NA
    vara_final = SEPARADOR_VALORES.join(varas) if varas else pd.NA

    return pd.Series([municipio_final, vara_final])

def encontrar_coluna(df, possibilidades):
    """
    Encontra uma coluna mesmo que venha com variação de acento,
    maiúscula/minúscula ou parte do nome.
    """

    mapa = {normalizar_texto(col): col for col in df.columns}

    for nome in possibilidades:
        nome_norm = normalizar_texto(nome)

        if nome_norm in mapa:
            return mapa[nome_norm]

    for col_norm, col_original in mapa.items():
        for nome in possibilidades:
            nome_norm = normalizar_texto(nome)

            if nome_norm in col_norm:
                return col_original

    return None


def valor_vazio(valor):
    """
    Verifica se um valor é vazio, nulo ou texto equivalente a vazio.
    """

    if pd.isna(valor):
        return True

    texto = str(valor).strip()

    if texto == "":
        return True

    if texto.lower() in ["nan", "none", "nat"]:
        return True

    return False


def limpar_valor(valor):
    """
    Limpa espaços duplicados e valores vazios.
    """

    if valor_vazio(valor):
        return ""

    texto = str(valor).strip()
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()


def limpar_nome(valor):
    """
    Limpa o nome da criança:
    - remove números
    - remove símbolos soltos no início/fim
    - remove espaços duplicados
    """

    if valor_vazio(valor):
        return pd.NA

    valor = str(valor)

    # Remove números
    valor = re.sub(r"\d+", "", valor)

    # Remove símbolos soltos no início/fim
    valor = re.sub(r"^[\s\-\–\—\.\,\;\:\|\/]+", "", valor)
    valor = re.sub(r"[\s\-\–\—\.\,\;\:\|\/]+$", "", valor)

    # Remove espaços duplicados
    valor = re.sub(r"\s+", " ", valor).strip()

    return valor if valor else pd.NA


def capitalizar_texto(valor):
    """
    Capitaliza textos preservando partículas como de, da, do, dos.
    """

    if valor_vazio(valor):
        return pd.NA

    particulas = {"de", "da", "do", "das", "dos", "e"}

    palavras = str(valor).strip().lower().split()
    palavras_formatadas = []

    for palavra in palavras:
        if palavra in particulas:
            palavras_formatadas.append(palavra)
        else:
            palavras_formatadas.append(palavra.capitalize())

    return " ".join(palavras_formatadas)


def capitalizar_nome(nome):
    """
    Capitaliza o nome da criança.
    """

    return capitalizar_texto(nome)


def chave_nome(nome):
    """
    Cria uma chave limpa para comparação de nomes.
    """

    nome = limpar_nome(nome)

    if valor_vazio(nome):
        return ""

    nome = normalizar_texto(nome)
    nome = re.sub(r"[^a-z\s]", " ", nome)
    nome = re.sub(r"\s+", " ", nome).strip()

    return nome


def tokens_nome(nome):
    """
    Quebra o nome em partes úteis para comparação.
    Remove partículas.
    """

    particulas = {"de", "da", "do", "das", "dos", "e"}

    return [
        token
        for token in chave_nome(nome).split()
        if token not in particulas
    ]


def similaridade_nome(nome_1, nome_2):
    """
    Calcula similaridade textual entre dois nomes.
    """

    return SequenceMatcher(
        None,
        chave_nome(nome_1),
        chave_nome(nome_2)
    ).ratio()


def chave_idade(valor):
    """
    Cria uma chave simples para idade.
    """

    if valor_vazio(valor):
        return ""

    texto = str(valor).strip()
    numeros = re.findall(r"\d+", texto)

    if numeros:
        return numeros[0]

    return normalizar_texto(texto)


def converter_idade_para_anos(valor):
    """
    Converte idade textual para anos inteiros.

    Regras:
    - 14 anos -> 14
    - 14 -> 14
    - 1 ano e 3 meses -> 1
    - 8 meses -> 0
    - 15 dias -> 0
    - vazio -> vazio
    """

    if valor_vazio(valor):
        return pd.NA

    texto_original = str(valor).strip()
    texto = normalizar_texto(texto_original)

    # Casos textuais de recém-nascido
    if texto in ["rn", "recem nascido", "recem nascida"]:
        return 0

    if "recem" in texto or "nascid" in texto:
        return 0

    tem_ano = bool(re.search(r"\bano\b|\banos\b", texto))
    tem_mes = bool(re.search(r"\bmes\b|\bmeses\b", texto))
    tem_dia = bool(re.search(r"\bdia\b|\bdias\b", texto))

    # Meses ou dias sem ano = menor de 1 ano
    if (tem_mes or tem_dia) and not tem_ano:
        return 0

    # Pega o número antes de ano/anos
    match_ano = re.search(r"(\d+)\s*(ano|anos)", texto)

    if match_ano:
        return int(match_ano.group(1))

    # Se for apenas número, considera anos
    numeros = re.findall(r"\d+", texto)

    if numeros:
        if tem_mes or tem_dia:
            return 0

        return int(numeros[0])

    return pd.NA


def converter_tempo_acolhimento_para_anos(valor):
    """
    Converte Tempo de Acolhimento para anos inteiros.

    Regras:
    - 4 ano(s), 2 mês(es) -> 4
    - 1 ano(s), 3 mês(es) -> 1
    - 9 mês(es), 28 dia(s) -> 0
    - 15 dia(s) -> 0
    - vazio -> vazio
    """

    if valor_vazio(valor):
        return pd.NA

    texto_original = str(valor).strip()
    texto = normalizar_texto(texto_original)

    match_anos = re.search(r"(\d+)\s*(ano|anos)", texto)

    if match_anos:
        return int(match_anos.group(1))

    tem_mes = bool(re.search(r"\bmes\b|\bmeses\b", texto))
    tem_dia = bool(re.search(r"\bdia\b|\bdias\b", texto))

    if tem_mes or tem_dia:
        return 0

    numeros = re.findall(r"\d+", texto)

    if numeros:
        return int(numeros[0])

    return pd.NA


def normalizar_processo(valor):
    """
    Normaliza números de processo/medida protetiva,
    mantendo apenas números.
    """

    if valor_vazio(valor):
        return ""

    valor = str(valor).strip()
    valor = re.sub(r"\D+", "", valor)

    return valor


def separar_processos(valor):
    """
    Separa processos quando estiverem unidos por |.
    """

    if valor_vazio(valor):
        return []

    texto = str(valor).strip()
    partes = re.split(r"\s*\|\s*", texto)

    processos = []

    for parte in partes:
        processo = normalizar_processo(parte)

        if processo != "":
            processos.append(processo)

    return processos


def juntar_valores(v1, v2):
    """
    Junta informações das duas planilhas sem perder dados.
    - Se forem iguais, mantém uma vez.
    - Se forem diferentes, junta com separador.
    """

    vazio_1 = valor_vazio(v1)
    vazio_2 = valor_vazio(v2)

    if vazio_1 and vazio_2:
        return pd.NA

    if vazio_1:
        return v2

    if vazio_2:
        return v1

    if normalizar_texto(v1) == normalizar_texto(v2):
        return v1

    valores = []

    for valor in [v1, v2]:
        partes = re.split(r"\s*\|\s*", str(valor))

        for parte in partes:
            parte_limpa = limpar_valor(parte)

            if parte_limpa == "":
                continue

            if normalizar_texto(parte_limpa) not in [normalizar_texto(v) for v in valores]:
                valores.append(parte_limpa)

    return SEPARADOR_VALORES.join(valores)


def combinar_serie(serie):
    """
    Combina valores duplicados dentro de uma mesma planilha.
    """

    valores = []

    for valor in serie:
        if valor_vazio(valor):
            continue

        partes = re.split(r"\s*\|\s*", str(valor))

        for parte in partes:
            parte_limpa = limpar_valor(parte)

            if parte_limpa == "":
                continue

            if normalizar_texto(parte_limpa) not in [normalizar_texto(v) for v in valores]:
                valores.append(parte_limpa)

    if len(valores) == 0:
        return pd.NA

    if len(valores) == 1:
        return valores[0]

    return SEPARADOR_VALORES.join(valores)


def escolher_nome_mais_completo(v1, v2):
    """
    Escolhe o nome mais completo entre dois nomes.
    """

    nomes = []

    for nome in [v1, v2]:
        if not valor_vazio(nome):
            nomes.append(nome)

    if len(nomes) == 0:
        return pd.NA

    nomes = sorted(
        nomes,
        key=lambda x: (len(tokens_nome(x)), len(str(x))),
        reverse=True
    )

    return nomes[0]


def combinar_nome_serie(serie):
    """
    Dentro de uma mesma planilha, mantém o nome mais completo.
    """

    nomes = [v for v in serie if not valor_vazio(v)]

    if len(nomes) == 0:
        return pd.NA

    nomes = sorted(
        nomes,
        key=lambda x: (len(tokens_nome(x)), len(str(x))),
        reverse=True
    )

    return nomes[0]


def nomes_possivelmente_mesma_pessoa(nome_1, nome_2, idade_1=None, idade_2=None):
    """
    Verifica se dois nomes podem ser da mesma pessoa.
    Usa idade quando disponível.
    """

    chave_1 = chave_nome(nome_1)
    chave_2 = chave_nome(nome_2)

    if chave_1 == "" or chave_2 == "":
        return False

    if idade_1 is not None and idade_2 is not None:
        idade_1_chave = chave_idade(idade_1)
        idade_2_chave = chave_idade(idade_2)

        if idade_1_chave != "" and idade_2_chave != "":
            if idade_1_chave != idade_2_chave:
                return False

    tokens_1 = set(tokens_nome(nome_1))
    tokens_2 = set(tokens_nome(nome_2))

    if tokens_1.issubset(tokens_2) or tokens_2.issubset(tokens_1):
        return True

    if chave_1 in chave_2 or chave_2 in chave_1:
        return True

    if similaridade_nome(nome_1, nome_2) >= 0.88:
        return True

    return False


def ajustar_situacao_e_detalhe(valor):
    """
    Garante que Situação fique apenas como Acolhido.
    O restante das informações vai para Detalhe.

    Exemplo:
    'Acolhido | Acolhimento com Prazo a Vencer'
    vira:
    Situação = Acolhido
    Detalhe = Acolhimento com Prazo a Vencer
    """

    if valor_vazio(valor):
        return pd.Series([SITUACAO_PADRAO, pd.NA])

    texto_original = str(valor).strip()
    partes = re.split(r"\s*\|\s*|\s*/\s*|\s*;\s*|\s*,\s*", texto_original)

    situacao_final = SITUACAO_PADRAO
    detalhes = []

    for parte in partes:
        parte_limpa = limpar_valor(parte)
        parte_norm = normalizar_texto(parte_limpa)

        if parte_norm == "":
            continue

        if parte_norm in ["acolhido", "acolhida"]:
            continue

        if "acolhimento" in parte_norm:
            detalhes.append(parte_limpa)
            continue

        detalhes.append(parte_limpa)

    detalhes_unicos = []

    for detalhe in detalhes:
        detalhe_norm = normalizar_texto(detalhe)

        if detalhe_norm in ["", "acolhido", "acolhida"]:
            continue

        if detalhe_norm not in [normalizar_texto(d) for d in detalhes_unicos]:
            detalhes_unicos.append(detalhe)

    detalhe_final = SEPARADOR_VALORES.join(detalhes_unicos) if detalhes_unicos else pd.NA

    return pd.Series([situacao_final, detalhe_final])


def juntar_detalhe_e_tipo(detalhe, tipo):
    """
    Move conteúdo da coluna Tipo para Detalhe sem repetir informação.
    """

    valores = []

    for valor in [detalhe, tipo]:
        if valor_vazio(valor):
            continue

        partes = re.split(r"\s*\|\s*", str(valor))

        for parte in partes:
            parte_limpa = limpar_valor(parte)

            if parte_limpa == "":
                continue

            if normalizar_texto(parte_limpa) not in [normalizar_texto(v) for v in valores]:
                valores.append(parte_limpa)

    if len(valores) == 0:
        return pd.NA

    return SEPARADOR_VALORES.join(valores)


def separar_orgao_julgador(valor):
    """
    Divide Órgão Julgador em Município e Unidade.

    Exemplo:
    'SALVADOR - 1ª VARA DA INFÂNCIA'
    vira:
    Município = Salvador
    Unidade = 1ª VARA DA INFÂNCIA
    """

    texto = limpar_valor(valor)

    if texto == "":
        return pd.Series([pd.NA, pd.NA])

    partes = re.split(r"\s*\|\s*", texto)

    municipios = []
    unidades = []

    for parte in partes:
        parte = limpar_valor(parte)

        if parte == "":
            continue

        split = re.split(r"\s*[-–—]\s*", parte, maxsplit=1)

        if len(split) == 1:
            municipio = split[0].strip()
            unidade = ""
        else:
            municipio = split[0].strip()
            unidade = split[1].strip()

        municipio = capitalizar_texto(municipio)

        if not valor_vazio(municipio):
            if normalizar_texto(municipio) not in [normalizar_texto(m) for m in municipios]:
                municipios.append(municipio)

        if not valor_vazio(unidade):
            if normalizar_texto(unidade) not in [normalizar_texto(u) for u in unidades]:
                unidades.append(unidade)

    municipio_final = SEPARADOR_VALORES.join(municipios) if municipios else pd.NA
    unidade_final = SEPARADOR_VALORES.join(unidades) if unidades else pd.NA

    return pd.Series([municipio_final, unidade_final])


print("Funções auxiliares carregadas com sucesso!")

In [ ]:
# ============================================================
# CÉLULA 4 - LEITURA DAS PLANILHAS
# ============================================================

from pathlib import Path
import os

# ============================================================
# 1. DEFINIR CAMINHOS DOS ARQUIVOS
# ============================================================

# Usa a pasta definida na Célula 1.
# Se por algum motivo ela não existir, usa a pasta atual.
try:
    PASTA_ARQUIVOS
except NameError:
    PASTA_ARQUIVOS = Path(".")

CAMINHO_ACOLHIMENTO_1 = PASTA_ARQUIVOS / ARQUIVO_ACOLHIMENTO_1
CAMINHO_ACOLHIMENTO_2 = PASTA_ARQUIVOS / ARQUIVO_ACOLHIMENTO_2

# ============================================================
# 2. VALIDAR SE OS ARQUIVOS EXISTEM NA PASTA
# ============================================================

arquivos_obrigatorios = [
    CAMINHO_ACOLHIMENTO_1,
    CAMINHO_ACOLHIMENTO_2
]

for caminho in arquivos_obrigatorios:
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}\n"
            f"Arquivos disponíveis na pasta: {os.listdir(PASTA_ARQUIVOS)}"
        )

# ============================================================
# 3. LER AS DUAS PLANILHAS DE ACOLHIMENTO
# ============================================================

df_acolhimento_1 = pd.read_excel(CAMINHO_ACOLHIMENTO_1)
df_acolhimento_2 = pd.read_excel(CAMINHO_ACOLHIMENTO_2)

# ============================================================
# 4. CONFERÊNCIA INICIAL
# ============================================================

print("Planilhas carregadas com sucesso!")

print("\nBase Acolhimento 1")
print(f"Arquivo: {CAMINHO_ACOLHIMENTO_1}")
print(f"Total de linhas: {len(df_acolhimento_1)}")
print(f"Total de colunas: {len(df_acolhimento_1.columns)}")
print("Colunas:")
print(df_acolhimento_1.columns.tolist())

print("\nBase Acolhimento 2")
print(f"Arquivo: {CAMINHO_ACOLHIMENTO_2}")
print(f"Total de linhas: {len(df_acolhimento_2)}")
print(f"Total de colunas: {len(df_acolhimento_2.columns)}")
print("Colunas:")
print(df_acolhimento_2.columns.tolist())

print("\nPrévia Acolhimento 1:")
display(df_acolhimento_1.head())

print("\nPrévia Acolhimento 2:")
display(df_acolhimento_2.head())

In [ ]:
# ============================================================
# CÉLULA 5 - PADRONIZAÇÃO DE COLUNAS, NOMES E ÓRGÃO JULGADOR
# ============================================================

def padronizar_base_acolhimento(df, nome_base, tipo_base):
    df = df.copy()

    # --------------------------------------------------------
    # 1. Identificar coluna de nome da criança
    # --------------------------------------------------------

    col_nome = encontrar_coluna(df, ["criança", "crianca", "nome"])

    if col_nome is None:
        raise ValueError(f"Não encontrei coluna de criança/nome na base {nome_base}.")

    # Renomeia para Criança
    if col_nome != "Criança":
        if "Criança" in df.columns:
            df["Criança"] = df["Criança"].fillna(df[col_nome])
            df = df.drop(columns=[col_nome])
        else:
            df = df.rename(columns={col_nome: "Criança"})

    # --------------------------------------------------------
    # 2. Limpar e capitalizar nomes
    # --------------------------------------------------------

    df["Criança"] = df["Criança"].apply(limpar_nome).apply(capitalizar_nome)

    # --------------------------------------------------------
    # 3. Padronizar coluna Situação
    # Mantém como veio. Se estiver vazia, preenche como Acolhido.
    # --------------------------------------------------------

    col_situacao = encontrar_coluna(df, ["situação", "situacao"])

    if col_situacao is None:
        df["Situação"] = SITUACAO_PADRAO
    else:
        if col_situacao != "Situação":
            df = df.rename(columns={col_situacao: "Situação"})

        df["Situação"] = df["Situação"].apply(
            lambda x: SITUACAO_PADRAO if valor_vazio(x) else limpar_valor(x)
        )

    # --------------------------------------------------------
    # 4. Padronizar coluna Idade
    # Mantém a idade original em texto para conversão correta depois
    # --------------------------------------------------------

    col_idade = encontrar_coluna(df, ["idade"])

    if col_idade is not None:
        if col_idade != "Idade":
            df = df.rename(columns={col_idade: "Idade"})

    # --------------------------------------------------------
    # 5. Separar Órgão Julgador de acordo com a origem
    # Planilha 1 -> Município Medida Protetiva / Vara Medida Protetiva
    # Planilha 2 -> Município Destituição / Vara Destituição
    # --------------------------------------------------------

    col_orgao = encontrar_coluna(df, ["órgão julgador", "orgao julgador"])

    if col_orgao is not None:
        if tipo_base == "medida_protetiva":
            df[["Município Medida Protetiva", "Vara Medida Protetiva"]] = (
                df[col_orgao].apply(separar_orgao_julgador_generico)
            )

        elif tipo_base == "destituicao":
            df[["Município Destituição", "Vara Destituição"]] = (
                df[col_orgao].apply(separar_orgao_julgador_generico)
            )

        else:
            raise ValueError(
                "tipo_base inválido. Use 'medida_protetiva' ou 'destituicao'."
            )

        # Remove Órgão Julgador depois de separar
        df = df.drop(columns=[col_orgao], errors="ignore")

    else:
        print(f"Atenção: coluna Órgão Julgador não encontrada em {nome_base}.")

    # --------------------------------------------------------
    # 6. Padronizar Tempo de Acolhimento
    # --------------------------------------------------------

    col_tempo = encontrar_coluna(df, ["tempo de acolhimento"])

    if col_tempo is not None:
        if col_tempo != "Tempo de Acolhimento":
            df = df.rename(columns={col_tempo: "Tempo de Acolhimento"})

    # --------------------------------------------------------
    # 7. Padronizar Número da Medida Protetiva
    # --------------------------------------------------------

    col_medida = encontrar_coluna(
        df,
        ["número da medida protetiva", "numero da medida protetiva"]
    )

    if col_medida is not None:
        if col_medida != "Número da medida protetiva":
            df = df.rename(columns={col_medida: "Número da medida protetiva"})

    # --------------------------------------------------------
    # 8. Padronizar Número da Destituição / Entrega Voluntária
    # --------------------------------------------------------

    col_destituicao = encontrar_coluna(
        df,
        [
            "número da destituição / entrega voluntária",
            "numero da destituicao / entrega voluntaria",
            "número da destituição",
            "numero da destituicao"
        ]
    )

    if col_destituicao is not None:
        if col_destituicao != "Número da destituição / entrega voluntária":
            df = df.rename(
                columns={
                    col_destituicao: "Número da destituição / entrega voluntária"
                }
            )

    # --------------------------------------------------------
    # 9. Padronizar coluna Tipo, caso exista
    # --------------------------------------------------------

    col_tipo = encontrar_coluna(df, ["tipo"])

    if col_tipo is not None:
        if col_tipo != "Tipo":
            df = df.rename(columns={col_tipo: "Tipo"})

    # --------------------------------------------------------
    # 10. Criar chaves auxiliares para desduplicação
    # --------------------------------------------------------

    df["__nome_chave__"] = df["Criança"].apply(chave_nome)

    if "Idade" in df.columns:
        df["__idade_chave__"] = df["Idade"].apply(chave_idade)
    else:
        df["__idade_chave__"] = ""

    df["__chave__"] = df.apply(
        lambda row: (
            f"{row['__nome_chave__']}||idade:{row['__idade_chave__']}"
            if row["__idade_chave__"] != ""
            else row["__nome_chave__"]
        ),
        axis=1
    )

    # Remove linhas sem nome de criança
    df = df[df["__nome_chave__"] != ""].copy()

    return df


# ============================================================
# APLICAR PADRONIZAÇÃO NAS DUAS BASES
# ============================================================

df_acolhimento_1_padronizado = padronizar_base_acolhimento(
    df_acolhimento_1,
    "Acolhimento 1",
    tipo_base="medida_protetiva"
)

df_acolhimento_2_padronizado = padronizar_base_acolhimento(
    df_acolhimento_2,
    "Acolhimento 2",
    tipo_base="destituicao"
)

# ============================================================
# CONFERÊNCIA
# ============================================================

print("Padronização concluída!")

print("\nAcolhimento 1 padronizado:")
print(f"Linhas: {len(df_acolhimento_1_padronizado)}")
print(f"Colunas: {len(df_acolhimento_1_padronizado.columns)}")
print(df_acolhimento_1_padronizado.columns.tolist())

print("\nAcolhimento 2 padronizado:")
print(f"Linhas: {len(df_acolhimento_2_padronizado)}")
print(f"Colunas: {len(df_acolhimento_2_padronizado.columns)}")
print(df_acolhimento_2_padronizado.columns.tolist())

print("\nPrévia Acolhimento 1:")
display(
    df_acolhimento_1_padronizado[
        [
            "Criança",
            "Situação",
            "Município Medida Protetiva",
            "Vara Medida Protetiva"
        ]
    ].head(20)
)

print("\nPrévia Acolhimento 2:")
display(
    df_acolhimento_2_padronizado[
        [
            "Criança",
            "Situação",
            "Município Destituição",
            "Vara Destituição"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# CÉLULA 6 - AUDITORIA DE NOMES PARECIDOS
# NÃO ALTERA NOMES AUTOMATICAMENTE
# ============================================================

df_acolhimento_1_corrigido = df_acolhimento_1_padronizado.copy()
df_acolhimento_2_corrigido = df_acolhimento_2_padronizado.copy()

# ============================================================
# 1. CRIAR BASE DE COMPARAÇÃO
# ============================================================

base_nomes_1 = df_acolhimento_1_corrigido[["Criança", "Idade"]].copy() if "Idade" in df_acolhimento_1_corrigido.columns else df_acolhimento_1_corrigido[["Criança"]].copy()
base_nomes_2 = df_acolhimento_2_corrigido[["Criança", "Idade"]].copy() if "Idade" in df_acolhimento_2_corrigido.columns else df_acolhimento_2_corrigido[["Criança"]].copy()

if "Idade" not in base_nomes_1.columns:
    base_nomes_1["Idade"] = pd.NA

if "Idade" not in base_nomes_2.columns:
    base_nomes_2["Idade"] = pd.NA

base_nomes_1["Origem"] = "Acolhimento 1"
base_nomes_2["Origem"] = "Acolhimento 2"

base_nomes = pd.concat(
    [base_nomes_1, base_nomes_2],
    ignore_index=True
)

base_nomes["__nome_chave__"] = base_nomes["Criança"].apply(chave_nome)
base_nomes["__idade_chave__"] = base_nomes["Idade"].apply(chave_idade)

base_nomes = base_nomes[
    base_nomes["__nome_chave__"] != ""
].drop_duplicates(
    subset=["Criança", "__idade_chave__", "Origem"]
).copy()

# ============================================================
# 2. GERAR APENAS AUDITORIA DE POSSÍVEIS NOMES PARECIDOS
# ============================================================

registros_auditoria = []

for idade, grupo in base_nomes.groupby("__idade_chave__"):
    registros = grupo.to_dict("records")

    for a, b in combinations(registros, 2):
        if a["Origem"] == b["Origem"]:
            continue

        nome_1 = a["Criança"]
        nome_2 = b["Criança"]

        chave_1 = chave_nome(nome_1)
        chave_2 = chave_nome(nome_2)

        if chave_1 == chave_2:
            continue

        tokens_1 = set(tokens_nome(nome_1))
        tokens_2 = set(tokens_nome(nome_2))

        if not tokens_1 or not tokens_2:
            continue

        sim = similaridade_nome(nome_1, nome_2)

        motivo = None

        # Apenas sugere casos de nome incompleto, sem corrigir automaticamente
        if tokens_1.issubset(tokens_2) or tokens_2.issubset(tokens_1):
            motivo = "Possível nome incompleto"

        elif sim >= 0.90:
            motivo = "Nome muito parecido - revisar manualmente"

        if motivo:
            registros_auditoria.append({
                "Idade": idade,
                "Nome Acolhimento 1": nome_1 if a["Origem"] == "Acolhimento 1" else nome_2,
                "Nome Acolhimento 2": nome_2 if b["Origem"] == "Acolhimento 2" else nome_1,
                "Similaridade": round(sim, 3),
                "Motivo": motivo
            })

auditoria_nomes = pd.DataFrame(registros_auditoria)

# ============================================================
# 3. CONFERÊNCIA
# ============================================================

print("Célula 6 concluída.")
print("Nenhum nome foi alterado automaticamente.")
print(f"Possíveis nomes parecidos para revisão: {len(auditoria_nomes)}")

if len(auditoria_nomes) > 0:
    display(auditoria_nomes)
else:
    print("Nenhum nome parecido encontrado para auditoria.")

print("\nAcolhimento 1 mantido:")
print(len(df_acolhimento_1_corrigido))

print("\nAcolhimento 2 mantido:")
print(len(df_acolhimento_2_corrigido))

In [ ]:
# ============================================================
# CÉLULA 7 - MESCLAGEM SEGURA DOS ACOLHIMENTOS 1 E 2
# Acolhimento 1 é a base principal
# A final preserva todas as linhas da planilha 1
# ============================================================

def criar_chave_mesclagem(row):
    nome = chave_nome(row.get("Criança", pd.NA))
    idade = chave_idade(row.get("Idade", pd.NA)) if "Idade" in row.index else ""

    if nome == "":
        return ""

    if idade != "":
        return f"{nome}||idade:{idade}"

    return nome


def combinar_base2_duplicada(df):
    """
    Se a base 2 tiver mais de um registro com a mesma chave,
    combina os dados dela antes de fazer o merge.
    """

    df = df.copy()
    df["__chave_merge__"] = df.apply(criar_chave_mesclagem, axis=1)

    df = df[df["__chave_merge__"] != ""].copy()

    agg = {}

    for col in df.columns:
        if col == "__chave_merge__":
            continue

        if col == "Criança":
            agg[col] = combinar_nome_serie
        else:
            agg[col] = combinar_serie

    return df.groupby("__chave_merge__", as_index=False).agg(agg)


# ============================================================
# 1. PREPARAR BASE 1, SEM DESDUPLICAR
# ============================================================

df_base1 = df_acolhimento_1_corrigido.copy()
df_base1["__chave_merge__"] = df_base1.apply(criar_chave_mesclagem, axis=1)

# Remove apenas linhas sem chave, se existirem
df_base1 = df_base1[df_base1["__chave_merge__"] != ""].copy()

# ============================================================
# 2. PREPARAR BASE 2, COMBINANDO DUPLICADOS INTERNOS
# ============================================================

df_base2 = combinar_base2_duplicada(df_acolhimento_2_corrigido)

# ============================================================
# 3. IDENTIFICAR REGISTROS DA BASE 2 QUE NÃO ENTRARÃO
# ============================================================

chaves_base1 = set(df_base1["__chave_merge__"])
chaves_base2 = set(df_base2["__chave_merge__"])

chaves_somente_base2 = chaves_base2 - chaves_base1

auditoria_somente_base2 = df_base2[
    df_base2["__chave_merge__"].isin(chaves_somente_base2)
].copy()

# ============================================================
# 4. PREPARAR MERGE
# ============================================================

df1_merge = df_base1.rename(
    columns={
        col: f"{col}__base1"
        for col in df_base1.columns
        if col != "__chave_merge__"
    }
)

df2_merge = df_base2.rename(
    columns={
        col: f"{col}__base2"
        for col in df_base2.columns
        if col != "__chave_merge__"
    }
)

# LEFT MERGE: mantém todas as linhas da planilha 1
df_mesclado = pd.merge(
    df1_merge,
    df2_merge,
    on="__chave_merge__",
    how="left"
)

# ============================================================
# 5. RECONSTRUIR A BASE FINAL
# ============================================================

colunas_base1 = [
    col.replace("__base1", "")
    for col in df_mesclado.columns
    if col.endswith("__base1")
]

colunas_base2 = [
    col.replace("__base2", "")
    for col in df_mesclado.columns
    if col.endswith("__base2")
]

todas_colunas = []

for col in colunas_base1 + colunas_base2:
    if col not in todas_colunas:
        todas_colunas.append(col)

df_acolhidos_desduplicado = pd.DataFrame()

for col in todas_colunas:
    col_1 = f"{col}__base1"
    col_2 = f"{col}__base2"

    # Criança e Idade sempre vêm da base 1
    if col in ["Criança", "Idade"]:
        if col_1 in df_mesclado.columns:
            df_acolhidos_desduplicado[col] = df_mesclado[col_1]
        elif col_2 in df_mesclado.columns:
            df_acolhidos_desduplicado[col] = df_mesclado[col_2]
        else:
            df_acolhidos_desduplicado[col] = pd.NA

    # Situação deve vir preferencialmente da base 2, se existir
    elif col == "Situação":
        df_acolhidos_desduplicado["Situação"] = df_mesclado.apply(
            lambda row: (
                row[col_2]
                if col_2 in df_mesclado.columns and not valor_vazio(row.get(col_2, pd.NA))
                else row[col_1]
                if col_1 in df_mesclado.columns and not valor_vazio(row.get(col_1, pd.NA))
                else SITUACAO_PADRAO
            ),
            axis=1
        )

    # Demais colunas combinam informações da base 1 e base 2
    else:
        df_acolhidos_desduplicado[col] = df_mesclado.apply(
            lambda row: juntar_valores(
                row[col_1] if col_1 in df_mesclado.columns else pd.NA,
                row[col_2] if col_2 in df_mesclado.columns else pd.NA
            ),
            axis=1
        )

# ============================================================
# 6. REMOVER COLUNAS AUXILIARES
# ============================================================

df_acolhidos_desduplicado = df_acolhidos_desduplicado.drop(
    columns=[
        "__nome_chave__",
        "__idade_chave__",
        "__chave__",
        "__chave_dedup__",
        "__chave_merge__"
    ],
    errors="ignore"
)

# ============================================================
# 7. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_acolhidos_desduplicado.columns
]

outras_colunas = [
    col for col in df_acolhidos_desduplicado.columns
    if col not in colunas_inicio
]

df_acolhidos_desduplicado = df_acolhidos_desduplicado[colunas_inicio + outras_colunas]

# ============================================================
# 8. CONFERÊNCIA
# ============================================================

print("Mesclagem segura concluída!")
print(f"Linhas da base 1 original: {len(df_acolhimento_1_corrigido)}")
print(f"Linhas finais após mesclagem: {len(df_acolhidos_desduplicado)}")

if len(df_acolhidos_desduplicado) == len(df_acolhimento_1_corrigido):
    print("OK: todas as crianças da planilha 1 foram preservadas.")
else:
    print("ATENÇÃO: a quantidade final ficou diferente da base 1.")

print("\nRegistros que existem somente na base 2 e não entraram na final:")
print(len(auditoria_somente_base2))

if len(auditoria_somente_base2) > 0:
    display(
        auditoria_somente_base2[
            [col for col in ["Criança", "Idade", "Situação"] if col in auditoria_somente_base2.columns]
        ]
    )

print("\nQuantidade por Situação:")
display(
    df_acolhidos_desduplicado["Situação"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "Situação", "Situação": "Quantidade"})
)

print("\nAmostra da base mesclada:")
display(df_acolhidos_desduplicado.head(20))

In [ ]:
# ============================================================
# CONFERÊNCIA DA MESCLAGEM ENTRE ACOLHIMENTO 1 E 2
# VERSÃO ATUALIZADA DA PIPELINE
# ============================================================

def criar_chave_conferencia(row):
    nome = chave_nome(row.get("Criança", pd.NA))
    idade = chave_idade(row.get("Idade", pd.NA)) if "Idade" in row.index else ""

    if nome == "":
        return ""

    if idade != "":
        return f"{nome}||idade:{idade}"

    return nome


# ============================================================
# 1. CRIAR CHAVES DE CONFERÊNCIA
# ============================================================

base1_conf = df_acolhimento_1_corrigido.copy()
base2_conf = df_acolhimento_2_corrigido.copy()

base1_conf["__chave_conf__"] = base1_conf.apply(criar_chave_conferencia, axis=1)
base2_conf["__chave_conf__"] = base2_conf.apply(criar_chave_conferencia, axis=1)

base1_conf = base1_conf[base1_conf["__chave_conf__"] != ""].copy()
base2_conf = base2_conf[base2_conf["__chave_conf__"] != ""].copy()

set_base1 = set(base1_conf["__chave_conf__"])
set_base2 = set(base2_conf["__chave_conf__"])

em_comum = set_base1 & set_base2
somente_base1 = set_base1 - set_base2
somente_base2 = set_base2 - set_base1

# ============================================================
# 2. CONFERÊNCIA DE QUANTIDADES
# ============================================================

print("Resumo da mesclagem atualizada:")
print(f"Linhas originais Acolhimento 1: {len(df_acolhimento_1_corrigido)}")
print(f"Linhas originais Acolhimento 2: {len(df_acolhimento_2_corrigido)}")
print(f"Total original somado: {len(df_acolhimento_1_corrigido) + len(df_acolhimento_2_corrigido)}")

print("\nChaves únicas:")
print(f"Crianças únicas no Acolhimento 1: {len(set_base1)}")
print(f"Crianças únicas no Acolhimento 2: {len(set_base2)}")

print("\nComparação entre as duas bases:")
print(f"Crianças que aparecem nas duas bases: {len(em_comum)}")
print(f"Crianças somente no Acolhimento 1: {len(somente_base1)}")
print(f"Crianças somente no Acolhimento 2: {len(somente_base2)}")

print("\nResultado da planilha final:")
print(f"Total esperado pela nova regra: {len(df_acolhimento_1_corrigido)}")
print(f"Total real da base final: {len(df_acolhidos_desduplicado)}")

if len(df_acolhidos_desduplicado) == len(df_acolhimento_1_corrigido):
    print("\nOK: a planilha final preservou todas as linhas do Acolhimento 1.")
else:
    print("\nATENÇÃO: a planilha final não tem a mesma quantidade da base Acolhimento 1.")

# ============================================================
# 3. MOSTRAR QUEM ESTÁ SOMENTE EM CADA BASE
# ============================================================

print("\nCrianças somente no Acolhimento 1:")
display(
    base1_conf[
        base1_conf["__chave_conf__"].isin(somente_base1)
    ][
        [col for col in ["Criança", "Idade", "Situação"] if col in base1_conf.columns]
    ].sort_values(by="Criança")
)

print("\nCrianças somente no Acolhimento 2:")
display(
    base2_conf[
        base2_conf["__chave_conf__"].isin(somente_base2)
    ][
        [col for col in ["Criança", "Idade", "Situação"] if col in base2_conf.columns]
    ].sort_values(by="Criança")
)

In [ ]:
# ============================================================
# CÉLULA 8 - PREPARAÇÃO DA BASE FINAL DE ACOLHIMENTO
# ============================================================

df_base_final = df_acolhidos_desduplicado.copy()

# ============================================================
# 1. MANTER SITUAÇÃO COMO VEIO
# ============================================================

if "Situação" not in df_base_final.columns:
    df_base_final["Situação"] = SITUACAO_PADRAO
else:
    df_base_final["Situação"] = df_base_final["Situação"].apply(
        lambda x: SITUACAO_PADRAO if valor_vazio(x) else limpar_valor(x)
    )

# ============================================================
# 2. REMOVER COLUNAS QUE NÃO DEVEM EXISTIR
# ============================================================

colunas_remover_agora = [
    "Origem Acolhimento",
    "Adoção",
    "Guarda"
]

df_base_final = df_base_final.drop(
    columns=[col for col in colunas_remover_agora if col in df_base_final.columns],
    errors="ignore"
)

# ============================================================
# 3. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_base_final.columns
]

outras_colunas = [
    col for col in df_base_final.columns
    if col not in colunas_inicio
]

df_base_final = df_base_final[colunas_inicio + outras_colunas]

# ============================================================
# 4. CONFERÊNCIA
# ============================================================

print("Base final de acolhimento preparada com sucesso!")
print("A coluna Situação foi mantida como veio nas planilhas.")
print(f"Total de linhas: {len(df_base_final)}")
print(f"Total de colunas: {len(df_base_final.columns)}")

print("\nValores encontrados na coluna Situação:")
display(
    df_base_final["Situação"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "Situação", "Situação": "Quantidade"})
)

print("\nAmostra da base final:")
display(df_base_final.head(20))

In [ ]:
# ============================================================
# CÉLULA 9 - CONSOLIDAÇÃO DA BASE FINAL CORRIGIDA
# Mantém Idade original e cria Idade (anos)
# Mantém Situação como veio
# Mantém colunas separadas de Município/Vara por origem
# Remove apenas colunas auxiliares
# ============================================================

df_final_completo = df_base_final.copy()

# ============================================================
# 1. GARANTIR COLUNA SITUAÇÃO, SEM SOBRESCREVER
# ============================================================

col_situacao = encontrar_coluna(df_final_completo, ["Situação", "Situacao"])

if col_situacao is None:
    df_final_completo["Situação"] = SITUACAO_PADRAO
else:
    if col_situacao != "Situação":
        df_final_completo = df_final_completo.rename(columns={col_situacao: "Situação"})

    # Mantém a Situação como veio, apenas limpa espaços
    df_final_completo["Situação"] = df_final_completo["Situação"].apply(
        lambda x: SITUACAO_PADRAO if valor_vazio(x) else limpar_valor(x)
    )

# ============================================================
# 2. GARANTIR COLUNA CRIANÇA
# ============================================================

if "Criança" not in df_final_completo.columns:
    raise ValueError("A coluna 'Criança' não foi encontrada.")

df_final_completo["Criança"] = (
    df_final_completo["Criança"]
    .apply(limpar_nome)
    .apply(capitalizar_nome)
)

df_final_completo = df_final_completo[
    df_final_completo["Criança"].notna()
].copy()

# ============================================================
# 3. MANTER IDADE ORIGINAL E CRIAR IDADE (ANOS)
# ============================================================

col_idade = encontrar_coluna(df_final_completo, ["Idade"])

if col_idade is not None:
    # Padroniza o nome da coluna original para Idade
    if col_idade != "Idade":
        df_final_completo = df_final_completo.rename(columns={col_idade: "Idade"})

    # Cria nova coluna sem apagar a original
    df_final_completo["Idade (anos)"] = (
        df_final_completo["Idade"]
        .apply(converter_idade_para_anos)
    )

else:
    if "Idade" not in df_final_completo.columns:
        df_final_completo["Idade"] = pd.NA

    if "Idade (anos)" not in df_final_completo.columns:
        df_final_completo["Idade (anos)"] = pd.NA

# ============================================================
# 4. GARANTIR COLUNAS DE MUNICÍPIO E VARA POR ORIGEM
# Essas colunas agora já devem ter sido criadas na Célula 5
# ============================================================

colunas_municipio_vara = [
    "Município Medida Protetiva",
    "Vara Medida Protetiva",
    "Município Destituição",
    "Vara Destituição"
]

for col in colunas_municipio_vara:
    if col not in df_final_completo.columns:
        df_final_completo[col] = pd.NA

# ============================================================
# 5. REMOVER COLUNAS ANTIGAS E AUXILIARES
# ============================================================

colunas_auxiliares = [
    col for col in df_final_completo.columns
    if str(col).startswith("__")
]

df_final_completo = df_final_completo.drop(
    columns=colunas_auxiliares,
    errors="ignore"
)

df_final_completo = df_final_completo.drop(
    columns=[
        "Origem Acolhimento",
        "Guarda",
        "Adoção",
        "Adocao",
        "Órgão Julgador",
        "Orgao Julgador",
        "Município",
        "Unidade"
    ],
    errors="ignore"
)

# ============================================================
# 6. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_final_completo.columns
]

outras_colunas = [
    col for col in df_final_completo.columns
    if col not in colunas_inicio
]

df_final_completo = df_final_completo[colunas_inicio + outras_colunas]

# Ordenar por nome da criança
df_final_completo = df_final_completo.sort_values(
    by="Criança",
    na_position="last"
).reset_index(drop=True)

# ============================================================
# 7. CONFERÊNCIA
# ============================================================

print("Base final consolidada corretamente!")
print(f"Total de linhas: {len(df_final_completo)}")
print(f"Total de colunas: {len(df_final_completo.columns)}")

print("\nColunas atuais:")
print(df_final_completo.columns.tolist())

print("\nQuantidade por Situação:")
display(
    df_final_completo["Situação"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "Situação", "Situação": "Quantidade"})
)

print("\nAmostra com Idade, Município e Vara:")
display(
    df_final_completo[
        [
            "Criança",
            "Situação",
            "Idade",
            "Idade (anos)",
            "Município Medida Protetiva",
            "Vara Medida Protetiva",
            "Município Destituição",
            "Vara Destituição"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# CÉLULA 10 - TRATAMENTO DA COLUNA SITUAÇÃO
# REGRA:
# - Manter Situação como veio
# - Não separar Situação
# - Não criar Detalhe
# - Remover Detalhe, caso exista
# ============================================================

df_final_completo = df_final_completo.copy()

# ============================================================
# 1. GARANTIR COLUNA SITUAÇÃO
# ============================================================

col_situacao = encontrar_coluna(df_final_completo, ["Situação", "Situacao"])

if col_situacao is None:
    df_final_completo["Situação"] = SITUACAO_PADRAO
else:
    if col_situacao != "Situação":
        df_final_completo = df_final_completo.rename(columns={col_situacao: "Situação"})

# ============================================================
# 2. LIMPAR ESPAÇOS, SEM SEPARAR O CONTEÚDO
# ============================================================

df_final_completo["Situação"] = df_final_completo["Situação"].apply(
    lambda x: SITUACAO_PADRAO if valor_vazio(x) else limpar_valor(x)
)

# ============================================================
# 3. REMOVER COLUNA DETALHE, CASO EXISTA
# ============================================================

col_detalhe = encontrar_coluna(df_final_completo, ["Detalhe"])

if col_detalhe is not None:
    df_final_completo = df_final_completo.drop(columns=[col_detalhe], errors="ignore")

# ============================================================
# 4. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_final_completo.columns
]

outras_colunas = [
    col for col in df_final_completo.columns
    if col not in colunas_inicio
]

df_final_completo = df_final_completo[colunas_inicio + outras_colunas]

# ============================================================
# 5. CONFERÊNCIA
# ============================================================

print("Tratamento da coluna Situação concluído!")
print("A coluna Situação foi mantida como veio.")
print("A coluna Detalhe foi removida, caso existisse.")

print("\nValores encontrados na coluna Situação:")
display(
    df_final_completo["Situação"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "Situação", "Situação": "Quantidade"})
)

print("\nAmostra:")
display(df_final_completo.head(30))

print("\nColunas atuais:")
print(df_final_completo.columns.tolist())

In [ ]:
# ============================================================
# CÉLULA 11 - CORREÇÃO DE IDADE (ANOS)
# Mantém a coluna Idade original
# Cria/corrige a coluna Idade (anos)
# Idades em meses ou dias viram 0
# Campos vazios continuam vazios
# ============================================================

df_final_completo = df_final_completo.copy()

# ============================================================
# 1. IDENTIFICAR COLUNA IDADE
# ============================================================

col_idade = encontrar_coluna(df_final_completo, ["Idade"])

if col_idade is None:
    raise ValueError("Não encontrei a coluna 'Idade' na planilha.")

if col_idade != "Idade":
    df_final_completo = df_final_completo.rename(columns={col_idade: "Idade"})

# Guarda idade anterior, caso já exista
if "Idade (anos)" in df_final_completo.columns:
    idade_anos_antes = df_final_completo["Idade (anos)"].copy()
else:
    idade_anos_antes = pd.Series([pd.NA] * len(df_final_completo))

# ============================================================
# 2. FUNÇÃO DE CONVERSÃO MAIS SEGURA
# ============================================================

def converter_idade_individual(valor):
    """
    Converte uma idade individual para anos.

    Regras:
    - '14 anos' -> 14
    - '14 ano(s), 2 mês(es)' -> 14
    - '1 ano(s), 5 mês(es)' -> 1
    - '8 meses' -> 0
    - '3 mês(es), 2 dia(s)' -> 0
    - '15 dias' -> 0
    - vazio -> vazio
    """

    if valor_vazio(valor):
        return pd.NA

    texto_original = str(valor).strip()

    if texto_original == "":
        return pd.NA

    texto = normalizar_texto(texto_original)

    # Casos textuais de recém-nascido
    if texto in ["rn", "recem nascido", "recem nascida"]:
        return 0

    if "recem" in texto or "nascid" in texto:
        return 0

    tem_ano = bool(re.search(r"\bano\b|\banos\b", texto))
    tem_mes = bool(re.search(r"\bmes\b|\bmeses\b", texto))
    tem_dia = bool(re.search(r"\bdia\b|\bdias\b", texto))

    # Se tem mês ou dia, mas não tem ano, é menor de 1 ano
    if (tem_mes or tem_dia) and not tem_ano:
        return 0

    # Se tiver ano, pega o número antes de ano/anos
    match_ano = re.search(r"(\d+)\s*(ano|anos)", texto)

    if match_ano:
        return int(match_ano.group(1))

    # Se for apenas número, considera anos
    numeros = re.findall(r"\d+", texto)

    if numeros:
        # Segurança: se caiu aqui mas tinha mês/dia, considera menor de 1 ano
        if tem_mes or tem_dia:
            return 0

        return int(numeros[0])

    return pd.NA


def converter_idade_final(valor):
    """
    Converte a coluna Idade.
    Se houver valores unidos por '|', converte todos.
    Se os valores forem diferentes, usa o último valor não vazio,
    pois normalmente vem da base mais atualizada.
    """

    if valor_vazio(valor):
        return pd.Series([pd.NA, "Vazio"])

    partes = re.split(r"\s*\|\s*", str(valor))

    conversoes = []

    for parte in partes:
        parte = limpar_valor(parte)

        if parte == "":
            continue

        idade_convertida = converter_idade_individual(parte)

        if not pd.isna(idade_convertida):
            conversoes.append(int(idade_convertida))

    if len(conversoes) == 0:
        return pd.Series([pd.NA, "Sem idade válida"])

    valores_unicos = []

    for idade in conversoes:
        if idade not in valores_unicos:
            valores_unicos.append(idade)

    if len(valores_unicos) == 1:
        return pd.Series([valores_unicos[0], "Convertido"])

    # Se houver conflito, usa o último valor encontrado
    return pd.Series([valores_unicos[-1], "Conflito de idade - usado último valor"])

# ============================================================
# 3. APLICAR CORREÇÃO
# ============================================================

resultado_idade = df_final_completo["Idade"].apply(converter_idade_final)

df_final_completo["Idade (anos)"] = resultado_idade[0]
df_final_completo["Status conversão idade"] = resultado_idade[1]

# ============================================================
# 4. CRIAR AUDITORIA DE IDADE
# ============================================================

auditoria_idade = pd.DataFrame({
    "Criança": df_final_completo["Criança"],
    "Idade original": df_final_completo["Idade"],
    "Idade (anos) antes": idade_anos_antes,
    "Idade (anos) corrigida": df_final_completo["Idade (anos)"],
    "Status conversão idade": df_final_completo["Status conversão idade"]
})

# Linhas com alteração ou conflito
auditoria_idade_alteracoes = auditoria_idade[
    (
        auditoria_idade["Idade (anos) antes"].astype(str).fillna("") !=
        auditoria_idade["Idade (anos) corrigida"].astype(str).fillna("")
    )
    |
    (
        auditoria_idade["Status conversão idade"].astype(str).str.contains("Conflito", case=False, na=False)
    )
].copy()

# ============================================================
# 5. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_final_completo.columns
]

outras_colunas = [
    col for col in df_final_completo.columns
    if col not in colunas_inicio
]

df_final_completo = df_final_completo[colunas_inicio + outras_colunas]

# ============================================================
# 6. CONFERÊNCIA
# ============================================================

print("Correção de Idade (anos) concluída!")
print("A coluna Idade original foi mantida.")
print("Crianças com idade em meses ou dias foram convertidas para 0.")
print("Campos de idade vazios permaneceram vazios.")

print(f"\nTotal de linhas: {len(df_final_completo)}")
print(f"Total de idades vazias: {df_final_completo['Idade (anos)'].isna().sum()}")
print(f"Total de alterações/conflitos auditados: {len(auditoria_idade_alteracoes)}")

print("\nContagem por Idade (anos):")
display(
    df_final_completo["Idade (anos)"]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
    .rename(columns={
        "index": "Idade (anos)",
        "Idade (anos)": "Quantidade"
    })
)

print("\nAmostra da correção:")
display(
    df_final_completo[
        ["Criança", "Idade", "Idade (anos)", "Status conversão idade"]
    ].head(30)
)

print("\nAuditoria de alterações/conflitos:")
display(auditoria_idade_alteracoes)

In [ ]:
# ============================================================
# CÉLULA 12 - CONFERÊNCIA DAS COLUNAS DE MUNICÍPIO E VARA
# ============================================================

df_final_completo = df_final_completo.copy()

colunas_esperadas = [
    "Município Medida Protetiva",
    "Vara Medida Protetiva",
    "Município Destituição",
    "Vara Destituição"
]

for col in colunas_esperadas:
    if col not in df_final_completo.columns:
        print(f"Atenção: coluna não encontrada: {col}")
    else:
        print(f"OK: {col}")

print("\nAmostra das colunas de Município e Vara:")
display(
    df_final_completo[
        [col for col in ["Criança"] + colunas_esperadas if col in df_final_completo.columns]
    ].head(30)
)

In [ ]:
# ============================================================
# CÉLULA 13 - CAPITALIZAÇÃO DAS COLUNAS DE MUNICÍPIO
# ============================================================

df_final_completo = df_final_completo.copy()

# ============================================================
# 1. CAPITALIZAR MUNICÍPIOS
# ============================================================

colunas_municipio = [
    "Município Medida Protetiva",
    "Município Destituição"
]

for col in colunas_municipio:
    if col in df_final_completo.columns:
        df_final_completo[col] = df_final_completo[col].apply(capitalizar_texto)
        print(f"Coluna capitalizada: {col}")
    else:
        print(f"Atenção: coluna não encontrada: {col}")

# ============================================================
# 2. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_final_completo.columns
]

outras_colunas = [
    col for col in df_final_completo.columns
    if col not in colunas_inicio
]

df_final_completo = df_final_completo[colunas_inicio + outras_colunas]

# ============================================================
# 3. CONFERÊNCIA
# ============================================================

print("\nCapitalização dos municípios concluída!")

print("\nAmostra das colunas de município:")
display(
    df_final_completo[
        [
            col for col in [
                "Criança",
                "Município Medida Protetiva",
                "Município Destituição"
            ]
            if col in df_final_completo.columns
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# CÉLULA 14 - CRIAÇÃO DE TEMPO DE ACOLHIMENTO (ANOS)
# ============================================================

df_final_completo = df_final_completo.copy()

# ============================================================
# 1. IDENTIFICAR COLUNA TEMPO DE ACOLHIMENTO
# ============================================================

col_tempo = encontrar_coluna(
    df_final_completo,
    ["Tempo de Acolhimento"]
)

if col_tempo is None:
    raise ValueError("Não encontrei a coluna 'Tempo de Acolhimento'.")

# Padroniza o nome da coluna original
if col_tempo != "Tempo de Acolhimento":
    df_final_completo = df_final_completo.rename(
        columns={col_tempo: "Tempo de Acolhimento"}
    )

# ============================================================
# 2. CRIAR COLUNA TEMPO DE ACOLHIMENTO (ANOS)
# ============================================================

df_final_completo["Tempo de Acolhimento (anos)"] = (
    df_final_completo["Tempo de Acolhimento"]
    .apply(converter_tempo_acolhimento_para_anos)
)

# ============================================================
# 3. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_final_completo.columns
]

outras_colunas = [
    col for col in df_final_completo.columns
    if col not in colunas_inicio
]

df_final_completo = df_final_completo[colunas_inicio + outras_colunas]

# ============================================================
# 4. CONFERÊNCIA
# ============================================================

print("Coluna 'Tempo de Acolhimento (anos)' criada com sucesso!")

print("\nAmostra da conversão:")
display(
    df_final_completo[
        ["Criança", "Tempo de Acolhimento", "Tempo de Acolhimento (anos)"]
    ].head(30)
)

print("\nContagem por Tempo de Acolhimento (anos):")
display(
    df_final_completo["Tempo de Acolhimento (anos)"]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
    .rename(columns={
        "index": "Tempo de Acolhimento (anos)",
        "Tempo de Acolhimento (anos)": "Quantidade"
    })
)

In [ ]:
# ============================================================
# CÉLULA 15 - REMOÇÃO DE COLUNAS AUXILIARES
# ============================================================

df_final_completo = df_final_completo.copy()

# ============================================================
# 1. LISTA DE COLUNAS QUE NÃO DEVEM IR PARA A PLANILHA FINAL
# ============================================================

colunas_remover = [
    "__chave__",
    "__chave_dedup__",
    "__nome_chave__",
    "__idade_chave__",
    "__origem__",
    "__linha_original__",
    "Origem Acolhimento",
    "Órgão Julgador",
    "Tipo",
    "Status conversão idade"
]

# Remove também qualquer coluna que comece com "__"
colunas_com_prefixo_auxiliar = [
    col for col in df_final_completo.columns
    if str(col).startswith("__")
]

colunas_remover_final = list(set(colunas_remover + colunas_com_prefixo_auxiliar))

df_final_completo = df_final_completo.drop(
    columns=[col for col in colunas_remover_final if col in df_final_completo.columns],
    errors="ignore"
)

# ============================================================
# 2. GARANTIR QUE COLUNAS IMPORTANTES NÃO FORAM REMOVIDAS
# ============================================================

colunas_obrigatorias = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",
    "Município Medida Protetiva",
    "Vara Medida Protetiva",
    "Município Destituição",
    "Vara Destituição"
]

for col in colunas_obrigatorias:
    if col not in df_final_completo.columns:
        print(f"Atenção: coluna obrigatória não encontrada: {col}")

# ============================================================
# 3. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_final_completo.columns
]

outras_colunas = [
    col for col in df_final_completo.columns
    if col not in colunas_inicio
]

df_final_completo = df_final_completo[colunas_inicio + outras_colunas]

# ============================================================
# 4. ORDENAR POR NOME DA CRIANÇA
# ============================================================

if "Criança" in df_final_completo.columns:
    df_final_completo = df_final_completo.sort_values(
        by="Criança",
        na_position="last"
    ).reset_index(drop=True)

# ============================================================
# 5. CONFERÊNCIA
# ============================================================

print("Colunas auxiliares removidas com sucesso!")
print(f"Total de linhas: {len(df_final_completo)}")
print(f"Total de colunas: {len(df_final_completo.columns)}")

print("\nColunas finais até aqui:")
print(df_final_completo.columns.tolist())

print("\nAmostra da planilha:")
display(df_final_completo.head(20))

In [ ]:
# ============================================================
# CÉLULA 15.1 - INSERIR ID INSTITUIÇÃO
# COM MAPA MANUAL + MUNICÍPIO + VARA
# ============================================================

import os
from pathlib import Path

df_final_completo = df_final_completo.copy()

# ============================================================
# 1. CONFIGURAR ARQUIVOS
# ============================================================

ARQUIVO_INSTITUICOES = "INSTITUICOES_PIPELINE.xlsx"
ARQUIVO_MAPA_MANUAL = "MAPA_ID_INSTITUICAO_MANUAL.xlsx"

try:
    PASTA_ARQUIVOS
except NameError:
    PASTA_ARQUIVOS = Path(".")

CAMINHO_INSTITUICOES = PASTA_ARQUIVOS / ARQUIVO_INSTITUICOES
CAMINHO_MAPA_MANUAL = PASTA_ARQUIVOS / ARQUIVO_MAPA_MANUAL

if not CAMINHO_INSTITUICOES.exists():
    raise FileNotFoundError(
        f"Arquivo de instituições não encontrado: {CAMINHO_INSTITUICOES}\n"
        f"Arquivos disponíveis na pasta: {os.listdir(PASTA_ARQUIVOS)}"
    )

# ============================================================
# 2. LER PLANILHA DE INSTITUIÇÕES
# ============================================================

df_instituicoes = pd.read_excel(CAMINHO_INSTITUICOES)

col_id_inst = encontrar_coluna(df_instituicoes, ["ID"])
col_nome_inst = encontrar_coluna(df_instituicoes, ["Instituição", "Instituicao"])
col_municipio_inst = encontrar_coluna(df_instituicoes, ["Município", "Municipio"])
col_vara_inst = encontrar_coluna(df_instituicoes, ["Vara"])

if col_id_inst is None:
    raise ValueError("Não encontrei a coluna 'ID' na planilha de instituições.")

if col_nome_inst is None:
    raise ValueError("Não encontrei a coluna 'Instituição' na planilha de instituições.")

if col_municipio_inst is None:
    raise ValueError("Não encontrei a coluna 'Município' na planilha de instituições.")

df_instituicoes = df_instituicoes.rename(
    columns={
        col_id_inst: "ID",
        col_nome_inst: "Instituição",
        col_municipio_inst: "Município"
    }
)

if col_vara_inst is not None and col_vara_inst != "Vara":
    df_instituicoes = df_instituicoes.rename(columns={col_vara_inst: "Vara"})

if "Vara" not in df_instituicoes.columns:
    df_instituicoes["Vara"] = pd.NA

# ============================================================
# 3. IDENTIFICAR COLUNAS NA PLANILHA FINAL
# ============================================================

col_servico = encontrar_coluna(
    df_final_completo,
    [
        "Serviço de Acolhimento",
        "Servico de Acolhimento",
        "Instituição",
        "Instituicao"
    ]
)

if col_servico is None:
    raise ValueError(
        "Não encontrei na planilha final uma coluna de instituição/serviço."
    )

colunas_municipio_final = [
    col for col in [
        "Município Medida Protetiva",
        "Município Destituição",
        "Município"
    ]
    if col in df_final_completo.columns
]

colunas_vara_final = [
    col for col in [
        "Vara Medida Protetiva",
        "Vara Destituição",
        "Vara"
    ]
    if col in df_final_completo.columns
]

if len(colunas_municipio_final) == 0:
    raise ValueError("Não encontrei nenhuma coluna de município na planilha final.")

if len(colunas_vara_final) == 0:
    print("Atenção: nenhuma coluna de Vara encontrada na planilha final.")

print(f"Coluna usada como instituição: {col_servico}")
print(f"Colunas usadas como município: {colunas_municipio_final}")
print(f"Colunas usadas como vara: {colunas_vara_final}")

# ============================================================
# 4. FUNÇÕES DE NORMALIZAÇÃO
# ============================================================

def chave_id_inst_texto(valor):
    if valor_vazio(valor):
        return ""

    texto = str(valor).strip()
    texto = re.sub(r"\s+", " ", texto)

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = texto.lower().strip()

    return texto


def remover_prefixo_familia_acolhedora(valor):
    if valor_vazio(valor):
        return ""

    texto = str(valor).strip()

    texto = re.sub(
        r"^\s*fam[ií]lia\s+acolhedora\s+de\s+",
        "",
        texto,
        flags=re.IGNORECASE
    )

    return texto.strip()


def separar_multiplos(valor):
    if valor_vazio(valor):
        return []

    partes = re.split(r"\s*\|\s*", str(valor))

    return [
        limpar_valor(parte)
        for parte in partes
        if not valor_vazio(parte)
    ]


def obter_valores_linha(row, colunas):
    valores = []

    for col in colunas:
        valor = row.get(col, pd.NA)

        for parte in separar_multiplos(valor):
            parte_limpa = limpar_valor(parte)

            if parte_limpa == "":
                continue

            if chave_id_inst_texto(parte_limpa) not in [
                chave_id_inst_texto(v) for v in valores
            ]:
                valores.append(parte_limpa)

    return valores

# ============================================================
# 5. PREPARAR CHAVES DA PLANILHA DE INSTITUIÇÕES
# ============================================================

df_instituicoes["__nome_chave__"] = df_instituicoes["Instituição"].apply(chave_id_inst_texto)
df_instituicoes["__municipio_chave__"] = df_instituicoes["Município"].apply(chave_id_inst_texto)
df_instituicoes["__vara_chave__"] = df_instituicoes["Vara"].apply(chave_id_inst_texto)

df_instituicoes["__familia_nome_chave__"] = df_instituicoes["Instituição"].apply(
    lambda x: chave_id_inst_texto(remover_prefixo_familia_acolhedora(x))
)

# ============================================================
# 6. CRIAR MAPAS AUTOMÁTICOS
# ============================================================

mapa_nome_municipio = {}
mapa_nome_vara = {}
mapa_familia_municipio = {}
mapa_familia_vara = {}
mapa_nome_simples = {}

for _, row in df_instituicoes.iterrows():
    id_inst = row["ID"]

    if valor_vazio(id_inst):
        continue

    nome_chave = row["__nome_chave__"]
    municipio_chave = row["__municipio_chave__"]
    vara_chave = row["__vara_chave__"]
    familia_chave = row["__familia_nome_chave__"]

    # Nome + Município
    if nome_chave and municipio_chave:
        chave = (nome_chave, municipio_chave)
        mapa_nome_municipio.setdefault(chave, [])

        if str(id_inst) not in [str(x) for x in mapa_nome_municipio[chave]]:
            mapa_nome_municipio[chave].append(id_inst)

    # Nome + Vara
    if nome_chave and vara_chave:
        chave = (nome_chave, vara_chave)
        mapa_nome_vara.setdefault(chave, [])

        if str(id_inst) not in [str(x) for x in mapa_nome_vara[chave]]:
            mapa_nome_vara[chave].append(id_inst)

    # Família Acolhedora + Município
    nome_original_norm = row["__nome_chave__"]

    if "familia acolhedora de" in nome_original_norm:
        if familia_chave and municipio_chave:
            chave = (familia_chave, municipio_chave)
            mapa_familia_municipio.setdefault(chave, [])

            if str(id_inst) not in [str(x) for x in mapa_familia_municipio[chave]]:
                mapa_familia_municipio[chave].append(id_inst)

        if familia_chave and vara_chave:
            chave = (familia_chave, vara_chave)
            mapa_familia_vara.setdefault(chave, [])

            if str(id_inst) not in [str(x) for x in mapa_familia_vara[chave]]:
                mapa_familia_vara[chave].append(id_inst)

    # Nome simples
    if nome_chave:
        mapa_nome_simples.setdefault(nome_chave, [])

        if str(id_inst) not in [str(x) for x in mapa_nome_simples[nome_chave]]:
            mapa_nome_simples[nome_chave].append(id_inst)

# ============================================================
# 7. LER MAPA MANUAL, SE EXISTIR
# ============================================================

mapa_manual = {}

if CAMINHO_MAPA_MANUAL.exists():
    df_mapa_manual = pd.read_excel(CAMINHO_MAPA_MANUAL)

    col_manual_servico = encontrar_coluna(
        df_mapa_manual,
        ["Serviço de Acolhimento", "Servico de Acolhimento", "Instituição", "Instituicao"]
    )

    col_manual_municipio = encontrar_coluna(df_mapa_manual, ["Município", "Municipio"])
    col_manual_vara = encontrar_coluna(df_mapa_manual, ["Vara"])
    col_manual_id = encontrar_coluna(df_mapa_manual, ["ID Instituição", "ID Instituicao", "ID"])

    if col_manual_servico is None or col_manual_id is None:
        raise ValueError(
            "O mapa manual precisa ter pelo menos as colunas "
            "'Serviço de Acolhimento' e 'ID Instituição'."
        )

    for _, row in df_mapa_manual.iterrows():
        servico_chave = chave_id_inst_texto(row[col_manual_servico])
        municipio_chave = chave_id_inst_texto(row[col_manual_municipio]) if col_manual_municipio else ""
        vara_chave = chave_id_inst_texto(row[col_manual_vara]) if col_manual_vara else ""
        id_inst = row[col_manual_id]

        if valor_vazio(servico_chave) or valor_vazio(id_inst):
            continue

        chave = (servico_chave, municipio_chave, vara_chave)
        mapa_manual[chave] = id_inst

    print(f"Mapa manual carregado: {len(mapa_manual)} regras.")
else:
    df_mapa_manual = pd.DataFrame()
    print("Mapa manual não encontrado. A pipeline seguirá apenas com regras automáticas.")

# ============================================================
# 8. FUNÇÃO DE RESOLUÇÃO
# ============================================================

auditoria_id_instituicao = []

def retornar_se_unico(ids, metodo, crianca, servico, referencia, status_ambiguidade):
    if len(ids) == 1:
        return ids[0], metodo

    if len(ids) > 1:
        auditoria_id_instituicao.append({
            "Criança": crianca,
            "Serviço de Acolhimento": servico,
            "Referência testada": referencia,
            "Status": status_ambiguidade,
            "IDs possíveis": SEPARADOR_VALORES.join(str(x) for x in ids)
        })

        return pd.NA, "Ambíguo"

    return None, None


def resolver_id(servico, municipios, varas, crianca):
    nome_chave = chave_id_inst_texto(servico)

    # --------------------------------------------------------
    # 1. MAPA MANUAL
    # --------------------------------------------------------

    for municipio in municipios:
        for vara in varas:
            chave = (
                nome_chave,
                chave_id_inst_texto(municipio),
                chave_id_inst_texto(vara)
            )

            if chave in mapa_manual:
                return mapa_manual[chave], "Mapa manual: Serviço + Município + Vara"

    for municipio in municipios:
        chave = (
            nome_chave,
            chave_id_inst_texto(municipio),
            ""
        )

        if chave in mapa_manual:
            return mapa_manual[chave], "Mapa manual: Serviço + Município"

    for vara in varas:
        chave = (
            nome_chave,
            "",
            chave_id_inst_texto(vara)
        )

        if chave in mapa_manual:
            return mapa_manual[chave], "Mapa manual: Serviço + Vara"

    chave = (nome_chave, "", "")

    if chave in mapa_manual:
        return mapa_manual[chave], "Mapa manual: Serviço"

    # --------------------------------------------------------
    # 2. NOME + MUNICÍPIO
    # --------------------------------------------------------

    for municipio in municipios:
        municipio_chave = chave_id_inst_texto(municipio)

        ids = mapa_nome_municipio.get((nome_chave, municipio_chave), [])

        id_resolvido, metodo = retornar_se_unico(
            ids,
            f"Instituição + Município ({municipio})",
            crianca,
            servico,
            municipio,
            "Ambíguo em Instituição + Município"
        )

        if metodo is not None:
            return id_resolvido, metodo

    # --------------------------------------------------------
    # 3. FAMÍLIA ACOLHEDORA + MUNICÍPIO
    # --------------------------------------------------------

    for municipio in municipios:
        municipio_chave = chave_id_inst_texto(municipio)

        ids = mapa_familia_municipio.get((nome_chave, municipio_chave), [])

        id_resolvido, metodo = retornar_se_unico(
            ids,
            f"Família Acolhedora + Município ({municipio})",
            crianca,
            servico,
            municipio,
            "Ambíguo em Família Acolhedora + Município"
        )

        if metodo is not None:
            return id_resolvido, metodo

    # --------------------------------------------------------
    # 4. NOME + VARA
    # --------------------------------------------------------

    for vara in varas:
        vara_chave = chave_id_inst_texto(vara)

        ids = mapa_nome_vara.get((nome_chave, vara_chave), [])

        id_resolvido, metodo = retornar_se_unico(
            ids,
            f"Instituição + Vara ({vara})",
            crianca,
            servico,
            vara,
            "Ambíguo em Instituição + Vara"
        )

        if metodo is not None:
            return id_resolvido, metodo

    # --------------------------------------------------------
    # 5. FAMÍLIA ACOLHEDORA + VARA
    # --------------------------------------------------------

    for vara in varas:
        vara_chave = chave_id_inst_texto(vara)

        ids = mapa_familia_vara.get((nome_chave, vara_chave), [])

        id_resolvido, metodo = retornar_se_unico(
            ids,
            f"Família Acolhedora + Vara ({vara})",
            crianca,
            servico,
            vara,
            "Ambíguo em Família Acolhedora + Vara"
        )

        if metodo is not None:
            return id_resolvido, metodo

    # --------------------------------------------------------
    # 6. NOME SIMPLES, APENAS SE FOR ÚNICO
    # --------------------------------------------------------

    ids = mapa_nome_simples.get(nome_chave, [])

    id_resolvido, metodo = retornar_se_unico(
        ids,
        "Nome único",
        crianca,
        servico,
        "Nome simples",
        "Ambíguo por nome simples"
    )

    if metodo is not None:
        return id_resolvido, metodo

    auditoria_id_instituicao.append({
        "Criança": crianca,
        "Serviço de Acolhimento": servico,
        "Referência testada": (
            f"Municípios: {SEPARADOR_VALORES.join(municipios)} | "
            f"Varas: {SEPARADOR_VALORES.join(varas)}"
        ),
        "Status": "Não encontrado",
        "IDs possíveis": pd.NA
    })

    return pd.NA, "Não encontrado"


def buscar_id_linha(row):
    servico = row[col_servico]
    crianca = row["Criança"] if "Criança" in row.index else pd.NA

    municipios = obter_valores_linha(row, colunas_municipio_final)
    varas = obter_valores_linha(row, colunas_vara_final)

    if valor_vazio(servico):
        auditoria_id_instituicao.append({
            "Criança": crianca,
            "Serviço de Acolhimento": servico,
            "Referência testada": pd.NA,
            "Status": "Serviço vazio",
            "IDs possíveis": pd.NA
        })

        return pd.Series([pd.NA, "Serviço vazio"])

    instituicoes = separar_multiplos(servico)

    ids_encontrados = []
    metodos = []

    for inst in instituicoes:
        id_inst, metodo = resolver_id(
            inst,
            municipios,
            varas,
            crianca
        )

        if not valor_vazio(id_inst):
            if str(id_inst) not in [str(x) for x in ids_encontrados]:
                ids_encontrados.append(id_inst)

            if metodo not in metodos:
                metodos.append(metodo)

    if len(ids_encontrados) == 0:
        return pd.Series([pd.NA, "Não encontrado ou ambíguo"])

    return pd.Series([
        SEPARADOR_VALORES.join(str(x) for x in ids_encontrados),
        SEPARADOR_VALORES.join(metodos)
    ])

# ============================================================
# 9. APLICAR BUSCA
# ============================================================

resultado_ids = df_final_completo.apply(buscar_id_linha, axis=1)

df_final_completo["ID Instituição"] = resultado_ids[0]
df_final_completo["Método ID Instituição"] = resultado_ids[1]

# ============================================================
# 10. REPOSICIONAR ID ANTES DO SERVIÇO DE ACOLHIMENTO
# ============================================================

colunas = df_final_completo.columns.tolist()

for col in ["ID Instituição", "Método ID Instituição"]:
    if col in colunas:
        colunas.remove(col)

posicao_servico = colunas.index(col_servico)

colunas = (
    colunas[:posicao_servico]
    + ["ID Instituição", "Método ID Instituição"]
    + colunas[posicao_servico:]
)

df_final_completo = df_final_completo[colunas]

# ============================================================
# 11. AUDITORIA
# ============================================================

auditoria_id_instituicao = pd.DataFrame(auditoria_id_instituicao)

if len(auditoria_id_instituicao) > 0:
    auditoria_id_instituicao = auditoria_id_instituicao.drop_duplicates()

# ============================================================
# 12. CONFERÊNCIA
# ============================================================

print("Coluna 'ID Instituição' criada.")
print("A busca usa: mapa manual, município, vara e nome único.")
print(f"Total de registros: {len(df_final_completo)}")
print(f"IDs encontrados: {df_final_completo['ID Instituição'].notna().sum()}")
print(f"IDs não encontrados ou ambíguos: {df_final_completo['ID Instituição'].isna().sum()}")

print("\nMétodos utilizados:")
display(
    df_final_completo["Método ID Instituição"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "Método", "Método ID Instituição": "Quantidade"})
)

print("\nAmostra:")
display(
    df_final_completo[
        [
            col for col in [
                "Criança",
                "ID Instituição",
                "Método ID Instituição",
                col_servico,
                "Município Medida Protetiva",
                "Vara Medida Protetiva",
                "Município Destituição",
                "Vara Destituição"
            ]
            if col in df_final_completo.columns
        ]
    ].head(30)
)

print("\nAuditoria de IDs não encontrados ou ambíguos:")
display(auditoria_id_instituicao.head(100))

In [ ]:
# ============================================================
# CÉLULA 15.2 - CORRIGIR ID DAS UNIDADES REGIONAIS
# Regra específica para:
# "Unidade de Acolhimento Regional para Criança e Adolescente"
# ============================================================

df_final_completo = df_final_completo.copy()

# ============================================================
# 1. GARANTIR QUE A PLANILHA DE INSTITUIÇÕES ESTÁ CARREGADA
# ============================================================

ARQUIVO_INSTITUICOES = "INSTITUICOES_PIPELINE.xlsx"

try:
    PASTA_ARQUIVOS
except NameError:
    from pathlib import Path
    PASTA_ARQUIVOS = Path(".")

CAMINHO_INSTITUICOES = PASTA_ARQUIVOS / ARQUIVO_INSTITUICOES

df_instituicoes_regional = pd.read_excel(CAMINHO_INSTITUICOES)

# ============================================================
# 2. IDENTIFICAR COLUNAS
# ============================================================

col_id_inst = encontrar_coluna(df_instituicoes_regional, ["ID"])
col_nome_inst = encontrar_coluna(df_instituicoes_regional, ["Instituição", "Instituicao"])
col_municipio_inst = encontrar_coluna(df_instituicoes_regional, ["Município", "Municipio"])

if col_id_inst is None:
    raise ValueError("Não encontrei a coluna ID na planilha de instituições.")

if col_nome_inst is None:
    raise ValueError("Não encontrei a coluna Instituição na planilha de instituições.")

if col_municipio_inst is None:
    raise ValueError("Não encontrei a coluna Município na planilha de instituições.")

df_instituicoes_regional = df_instituicoes_regional.rename(
    columns={
        col_id_inst: "ID",
        col_nome_inst: "Instituição",
        col_municipio_inst: "Município"
    }
)

col_servico = encontrar_coluna(
    df_final_completo,
    [
        "Serviço de Acolhimento",
        "Servico de Acolhimento",
        "Instituição",
        "Instituicao"
    ]
)

if col_servico is None:
    raise ValueError("Não encontrei a coluna Serviço de Acolhimento na planilha final.")

# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def chave_regional(valor):
    if valor_vazio(valor):
        return ""

    texto = str(valor).strip()
    texto = re.sub(r"\s+", " ", texto)

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = texto.lower().strip()

    return texto


def pegar_primeiro_valor_valido(row, colunas):
    for col in colunas:
        if col in row.index:
            valor = row[col]

            if not valor_vazio(valor):
                partes = re.split(r"\s*\|\s*", str(valor))

                for parte in partes:
                    parte = limpar_valor(parte)

                    if parte != "":
                        return parte

    return pd.NA

# ============================================================
# 4. MAPA DO MUNICÍPIO DO PROCESSO PARA A UNIDADE REGIONAL
# ============================================================
# Ajuste este mapa sempre que aparecerem novos municípios regionais.

MAPA_MUNICIPIO_PARA_REGIONAL = {
    # Regional de Cícero Dantas - ID 55
    "cipo": "CICERO DANTAS",
    "cicero dantas": "CICERO DANTAS",
    "jeremoabo": "CICERO DANTAS",
    "antas": "CICERO DANTAS",

    # Regional de Lapão - ID 140
    "lapao": "LAPAO",
    "irece": "LAPAO",
    "joao dourado": "LAPAO",
    "canarana": "LAPAO",

    # Regional de Amargosa - ID 2
    "amargosa": "AMARGOSA",
    "maracas": "AMARGOSA",
    "mutuipe": "AMARGOSA",
    "santa teresinha": "AMARGOSA",
    "iacu": "AMARGOSA"
}

NOME_UNIDADE_REGIONAL = "Unidade de Acolhimento Regional para Criança e Adolescente"
CHAVE_UNIDADE_REGIONAL = chave_regional(NOME_UNIDADE_REGIONAL)

# ============================================================
# 5. CRIAR MAPA: UNIDADE REGIONAL + MUNICÍPIO REGIONAL -> ID
# ============================================================

df_regionais = df_instituicoes_regional[
    df_instituicoes_regional["Instituição"].apply(chave_regional) == CHAVE_UNIDADE_REGIONAL
].copy()

mapa_regional_id = {}

for _, row in df_regionais.iterrows():
    municipio_regional = chave_regional(row["Município"])
    id_inst = row["ID"]

    if municipio_regional != "" and not valor_vazio(id_inst):
        mapa_regional_id[municipio_regional] = id_inst

print("Regionais encontradas na planilha de instituições:")
display(df_regionais[["ID", "Instituição", "Município"]])

# ============================================================
# 6. APLICAR CORREÇÃO NOS IDs VAZIOS
# ============================================================

colunas_municipio_final = [
    "Município Destituição",
    "Município Medida Protetiva",
    "Município"
]

corrigidos = []

for idx, row in df_final_completo.iterrows():
    id_atual = row.get("ID Instituição", pd.NA)
    servico = row.get(col_servico, pd.NA)

    # Só corrige quem está vazio e é Unidade Regional
    if not valor_vazio(id_atual):
        continue

    if chave_regional(servico) != CHAVE_UNIDADE_REGIONAL:
        continue

    municipio_processo = pegar_primeiro_valor_valido(row, colunas_municipio_final)

    municipio_processo_chave = chave_regional(municipio_processo)

    municipio_regional = MAPA_MUNICIPIO_PARA_REGIONAL.get(municipio_processo_chave)

    if municipio_regional is None:
        corrigidos.append({
            "Linha": idx + 2,
            "Criança": row["Criança"] if "Criança" in row.index else pd.NA,
            "Serviço de Acolhimento": servico,
            "Município do processo": municipio_processo,
            "Município regional": pd.NA,
            "ID encontrado": pd.NA,
            "Status": "Município não está no mapa manual"
        })
        continue

    id_regional = mapa_regional_id.get(chave_regional(municipio_regional))

    if valor_vazio(id_regional):
        corrigidos.append({
            "Linha": idx + 2,
            "Criança": row["Criança"] if "Criança" in row.index else pd.NA,
            "Serviço de Acolhimento": servico,
            "Município do processo": municipio_processo,
            "Município regional": municipio_regional,
            "ID encontrado": pd.NA,
            "Status": "Município regional não encontrado na planilha de instituições"
        })
        continue

    df_final_completo.loc[idx, "ID Instituição"] = id_regional
    df_final_completo.loc[idx, "Método ID Instituição"] = (
        f"Mapa regional: {municipio_processo} -> {municipio_regional}"
    )

    corrigidos.append({
        "Linha": idx + 2,
        "Criança": row["Criança"] if "Criança" in row.index else pd.NA,
        "Serviço de Acolhimento": servico,
        "Município do processo": municipio_processo,
        "Município regional": municipio_regional,
        "ID encontrado": id_regional,
        "Status": "Corrigido"
    })

auditoria_regionais = pd.DataFrame(corrigidos)

# ============================================================
# 7. CONFERÊNCIA
# ============================================================

print("Correção das Unidades Regionais concluída!")

if len(auditoria_regionais) > 0:
    print(f"Total analisado/corrigido: {len(auditoria_regionais)}")
    display(auditoria_regionais)
else:
    print("Nenhuma Unidade Regional com ID vazio foi encontrada para corrigir.")

print("\nIDs ainda vazios para Unidade Regional:")
display(
    df_final_completo[
        df_final_completo["ID Instituição"].isna() &
        (df_final_completo[col_servico].apply(chave_regional) == CHAVE_UNIDADE_REGIONAL)
    ][
        [
            col for col in [
                "Criança",
                "ID Instituição",
                "Método ID Instituição",
                col_servico,
                "Município Medida Protetiva",
                "Município Destituição",
                "Vara Medida Protetiva",
                "Vara Destituição"
            ]
            if col in df_final_completo.columns
        ]
    ]
)

In [ ]:
# ============================================================
# CÉLULA 15.3 - REMOVER COLUNA "MÉTODO ID INSTITUIÇÃO"
# ============================================================

df_final_completo = df_final_completo.copy()

# Remove a coluna Método ID Instituição, caso exista
df_final_completo = df_final_completo.drop(
    columns=["Método ID Instituição"],
    errors="ignore"
)

# ============================================================
# ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)",

    "ID Instituição",
    "Serviço de Acolhimento"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_final_completo.columns
]

outras_colunas = [
    col for col in df_final_completo.columns
    if col not in colunas_inicio
]

df_final_completo = df_final_completo[colunas_inicio + outras_colunas]

# ============================================================
# CONFERÊNCIA
# ============================================================

print("Coluna 'Método ID Instituição' removida com sucesso!")

print("\nColunas atuais:")
print(df_final_completo.columns.tolist())

print("\nAmostra:")
display(df_final_completo.head(20))

In [ ]:
# ============================================================
# CÉLULA 16 - VALIDAÇÕES E AUDITORIAS
# ============================================================

df_validacao = df_final_completo.copy()

# ============================================================
# 1. VALIDAÇÕES GERAIS
# ============================================================

validacoes = []

def adicionar_validacao(item, resultado, observacao=""):
    validacoes.append({
        "Item": item,
        "Resultado": resultado,
        "Observação": observacao
    })

# Total geral
adicionar_validacao(
    "Total de linhas",
    len(df_validacao),
    "Quantidade total de registros na planilha final"
)

adicionar_validacao(
    "Total de colunas",
    len(df_validacao.columns),
    "Quantidade total de colunas na planilha final"
)

# Colunas obrigatórias
colunas_obrigatorias = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",
    "Município Medida Protetiva",
    "Vara Medida Protetiva",
    "Município Destituição",
    "Vara Destituição"
]

for col in colunas_obrigatorias:
    adicionar_validacao(
        f"Coluna obrigatória: {col}",
        "OK" if col in df_validacao.columns else "FALTANDO",
        ""
    )

# Situação
if "Situação" in df_validacao.columns:
    situacoes_encontradas = sorted(
        df_validacao["Situação"]
        .dropna()
        .astype(str)
        .unique()
    )

    adicionar_validacao(
        "Situações encontradas",
        ", ".join(situacoes_encontradas),
        "Situação mantida como veio nas bases originais"
    )

    situacao_vazia = df_validacao[
        df_validacao["Situação"].apply(valor_vazio)
    ]

    adicionar_validacao(
        "Situação vazia",
        len(situacao_vazia),
        "Registros sem informação na coluna Situação"
    )

# Nomes vazios
if "Criança" in df_validacao.columns:
    nomes_vazios = df_validacao[df_validacao["Criança"].isna()]

    adicionar_validacao(
        "Crianças sem nome",
        len(nomes_vazios),
        "Deve ser 0"
    )

# Idades
if "Idade" in df_validacao.columns and "Idade (anos)" in df_validacao.columns:
    idade_original_vazia = df_validacao["Idade"].apply(valor_vazio)

    idade_anos_vazia = df_validacao["Idade (anos)"].isna()

    idade_preenchida_com_original_vazia = df_validacao[
        idade_original_vazia & ~idade_anos_vazia
    ]

    adicionar_validacao(
        "Idade original vazia, mas Idade (anos) preenchida",
        len(idade_preenchida_com_original_vazia),
        "Deve ser 0. Se aparecer valor, revisar conversão."
    )

    idade_original_preenchida_sem_anos = df_validacao[
        ~idade_original_vazia & idade_anos_vazia
    ]

    adicionar_validacao(
        "Idade original preenchida, mas Idade (anos) vazia",
        len(idade_original_preenchida_sem_anos),
        "Revisar se há texto não reconhecido na idade."
    )

    idades_negativas = df_validacao[
        pd.to_numeric(df_validacao["Idade (anos)"], errors="coerce") < 0
    ]

    adicionar_validacao(
        "Idades negativas",
        len(idades_negativas),
        "Deve ser 0"
    )

# Município e Vara vazios
colunas_municipio_vara = [
    "Município Medida Protetiva",
    "Vara Medida Protetiva",
    "Município Destituição",
    "Vara Destituição"
]

for col in colunas_municipio_vara:
    if col in df_validacao.columns:
        vazios = df_validacao[
            df_validacao[col].apply(valor_vazio)
        ]

        adicionar_validacao(
            f"{col} vazio",
            len(vazios),
            f"Registros sem informação em {col}"
        )

resumo_validacoes = pd.DataFrame(validacoes)

# ============================================================
# 2. RESUMO POR SITUAÇÃO
# ============================================================

if "Situação" in df_validacao.columns:
    resumo_situacao = (
        df_validacao["Situação"]
        .value_counts(dropna=False)
        .reset_index()
    )

    resumo_situacao.columns = ["Situação", "Quantidade"]
else:
    resumo_situacao = pd.DataFrame(columns=["Situação", "Quantidade"])

# ============================================================
# 3. RESUMO POR IDADE
# ============================================================

if "Idade (anos)" in df_validacao.columns:
    resumo_idade = (
        df_validacao["Idade (anos)"]
        .value_counts(dropna=False)
        .sort_index()
        .reset_index()
    )

    resumo_idade.columns = ["Idade (anos)", "Quantidade"]
else:
    resumo_idade = pd.DataFrame(columns=["Idade (anos)", "Quantidade"])

# ============================================================
# 4. CONTAGEM DA COLUNA NÚMERO DA MEDIDA PROTETIVA
# ============================================================

col_medida = encontrar_coluna(
    df_validacao,
    ["Número da medida protetiva", "Numero da medida protetiva"]
)

if col_medida is not None:
    medidas_lista = []

    for idx, valor in df_validacao[col_medida].items():
        processos = separar_processos(valor)

        for processo in processos:
            medidas_lista.append({
                "Linha": idx + 2,
                "Número da medida protetiva": processo,
                "Criança": df_validacao.loc[idx, "Criança"] if "Criança" in df_validacao.columns else pd.NA
            })

    df_medidas_explodido = pd.DataFrame(medidas_lista)

    if len(df_medidas_explodido) > 0:
        count_medidas = (
            df_medidas_explodido["Número da medida protetiva"]
            .value_counts()
            .reset_index()
        )

        count_medidas.columns = ["Número da medida protetiva", "Quantidade"]

        repetidos_medida_protetiva = count_medidas[
            count_medidas["Quantidade"] > 1
        ].copy()

        repetidos_medida_protetiva = repetidos_medida_protetiva.merge(
            df_medidas_explodido.groupby("Número da medida protetiva")["Criança"]
            .apply(lambda x: SEPARADOR_VALORES.join(sorted(set(x.dropna().astype(str)))))
            .reset_index(),
            on="Número da medida protetiva",
            how="left"
        )

    else:
        count_medidas = pd.DataFrame(columns=["Número da medida protetiva", "Quantidade"])
        repetidos_medida_protetiva = pd.DataFrame(columns=["Número da medida protetiva", "Quantidade", "Criança"])

else:
    df_medidas_explodido = pd.DataFrame(columns=["Linha", "Número da medida protetiva", "Criança"])
    count_medidas = pd.DataFrame(columns=["Número da medida protetiva", "Quantidade"])
    repetidos_medida_protetiva = pd.DataFrame(columns=["Número da medida protetiva", "Quantidade", "Criança"])

# ============================================================
# 5. POSSÍVEIS INCONSISTÊNCIAS DE NOMES
# ============================================================

def analisar_inconsistencia_nome(nome_1, nome_2):
    tokens_1 = tokens_nome(nome_1)
    tokens_2 = tokens_nome(nome_2)

    if not tokens_1 or not tokens_2:
        return None

    if chave_nome(nome_1) == chave_nome(nome_2):
        return None

    set_1 = set(tokens_1)
    set_2 = set(tokens_2)

    sobrenomes_1 = set(tokens_1[1:])
    sobrenomes_2 = set(tokens_2[1:])

    intersecao = set_1 & set_2
    sobrenomes_iguais = sobrenomes_1 & sobrenomes_2

    sim_total = similaridade_nome(nome_1, nome_2)

    primeiro_1 = tokens_1[0]
    primeiro_2 = tokens_2[0]

    sim_primeiro = SequenceMatcher(None, primeiro_1, primeiro_2).ratio()

    if set_1.issubset(set_2) or set_2.issubset(set_1):
        return "Nome possivelmente incompleto"

    if len(sobrenomes_iguais) >= 2 and sim_primeiro >= 0.70:
        return "Possível erro no primeiro nome com sobrenomes iguais"

    if sim_total >= 0.88 and len(intersecao) >= 2:
        return "Nome muito parecido"

    if len(intersecao) >= 3:
        return "Muitos termos iguais no nome"

    return None


registros_inconsistencias = []

if "Criança" in df_validacao.columns:
    base_nomes_validacao = df_validacao.copy()

    base_nomes_validacao["__nome_chave_validacao__"] = base_nomes_validacao["Criança"].apply(chave_nome)

    if "Idade (anos)" in base_nomes_validacao.columns:
        base_nomes_validacao["__idade_validacao__"] = base_nomes_validacao["Idade (anos)"].fillna("SEM IDADE")
    else:
        base_nomes_validacao["__idade_validacao__"] = "SEM IDADE"

    base_nomes_validacao = base_nomes_validacao[
        base_nomes_validacao["__nome_chave_validacao__"] != ""
    ].copy()

    for idade, grupo in base_nomes_validacao.groupby("__idade_validacao__"):
        registros = grupo.to_dict("records")

        for a, b in combinations(registros, 2):
            nome_1 = a["Criança"]
            nome_2 = b["Criança"]

            motivo = analisar_inconsistencia_nome(nome_1, nome_2)

            if motivo:
                registros_inconsistencias.append({
                    "Idade (anos)": idade,
                    "Nome 1": nome_1,
                    "Nome 2": nome_2,
                    "Similaridade": round(similaridade_nome(nome_1, nome_2), 3),
                    "Motivo": motivo
                })

auditoria_nomes_possiveis = pd.DataFrame(registros_inconsistencias)

if len(auditoria_nomes_possiveis) > 0:
    auditoria_nomes_possiveis = auditoria_nomes_possiveis.sort_values(
        by=["Motivo", "Idade (anos)", "Nome 1", "Nome 2"]
    )

# Se a auditoria da célula 6 existir, mantém também
try:
    auditoria_nomes_corrigidos = auditoria_nomes.copy()
except NameError:
    auditoria_nomes_corrigidos = pd.DataFrame(
        columns=["Nome antes", "Idade chave", "Nome corrigido"]
    )

# Se a auditoria da célula 11 existir, mantém também
try:
    auditoria_idade_final = auditoria_idade_alteracoes.copy()
except NameError:
    auditoria_idade_final = pd.DataFrame(
        columns=[
            "Criança",
            "Idade original",
            "Idade (anos) antes",
            "Idade (anos) corrigida",
            "Status conversão idade"
        ]
    )

# ============================================================
# 6. RESUMO FINAL DA PIPELINE
# ============================================================

resumo_pipeline = pd.DataFrame([
    {
        "Indicador": "Total de registros finais",
        "Valor": len(df_final_completo)
    },
    {
        "Indicador": "Total de colunas finais",
        "Valor": len(df_final_completo.columns)
    },
    {
        "Indicador": "Total de nomes corrigidos/completados",
        "Valor": len(auditoria_nomes_corrigidos)
    },
    {
        "Indicador": "Possíveis inconsistências de nomes",
        "Valor": len(auditoria_nomes_possiveis)
    },
    {
        "Indicador": "Total de idades alteradas/conflitantes",
        "Valor": len(auditoria_idade_final)
    },
    {
        "Indicador": "Medidas protetivas preenchidas",
        "Valor": len(df_medidas_explodido)
    },
    {
        "Indicador": "Medidas protetivas únicas",
        "Valor": df_medidas_explodido["Número da medida protetiva"].nunique() if len(df_medidas_explodido) > 0 else 0
    },
    {
        "Indicador": "Medidas protetivas repetidas",
        "Valor": len(repetidos_medida_protetiva)
    }
])

# ============================================================
# 7. EXIBIR RESULTADOS
# ============================================================

print("Validações e auditorias concluídas!")

print("\nResumo da pipeline:")
display(resumo_pipeline)

print("\nResumo das validações:")
display(resumo_validacoes)

print("\nResumo por Situação:")
display(resumo_situacao)

print("\nResumo por Idade:")
display(resumo_idade)

print("\nMedidas protetivas repetidas:")
display(repetidos_medida_protetiva)

print("\nPossíveis inconsistências de nomes:")
display(auditoria_nomes_possiveis)

print("\nAuditoria de nomes corrigidos/completados:")
display(auditoria_nomes_corrigidos)

print("\nAuditoria de idade:")
display(auditoria_idade_final)

In [ ]:
# ============================================================
# CÉLULA 17 - EXPORTAÇÃO FINAL FORMATADA
# ============================================================

df_exportar = df_final_completo.copy()

# ============================================================
# 1. GARANTIR QUE COLUNAS AUXILIARES NÃO SERÃO EXPORTADAS
# ============================================================

colunas_auxiliares_exportacao = [
    col for col in df_exportar.columns
    if str(col).startswith("__")
]

df_exportar = df_exportar.drop(
    columns=colunas_auxiliares_exportacao,
    errors="ignore"
)

# Remove colunas técnicas, caso ainda existam
df_exportar = df_exportar.drop(
    columns=[
        "Status conversão idade",
        "Origem Acolhimento",
        "Tipo",
        "Órgão Julgador"
    ],
    errors="ignore"
)

# ============================================================
# 2. ORGANIZAR COLUNAS PRINCIPAIS
# ============================================================

COLUNAS_INICIO = [
    "Criança",
    "Situação",
    "Idade",
    "Idade (anos)",

    "Município Medida Protetiva",
    "Vara Medida Protetiva",

    "Município Destituição",
    "Vara Destituição",

    "Tempo de Acolhimento",
    "Tempo de Acolhimento (anos)"
]

colunas_inicio = [
    col for col in COLUNAS_INICIO
    if col in df_exportar.columns
]

outras_colunas = [
    col for col in df_exportar.columns
    if col not in colunas_inicio
]

df_exportar = df_exportar[colunas_inicio + outras_colunas]

# Ordenar por criança
if "Criança" in df_exportar.columns:
    df_exportar = df_exportar.sort_values(
        by="Criança",
        na_position="last"
    ).reset_index(drop=True)

# ============================================================
# 3. AJUSTAR COLUNAS NUMÉRICAS
# ============================================================

for col in ["Idade (anos)", "Tempo de Acolhimento (anos)"]:
    if col in df_exportar.columns:
        df_exportar[col] = pd.to_numeric(
            df_exportar[col],
            errors="coerce"
        ).astype("Int64")

# ============================================================
# 4. GARANTIR DATAFRAMES DAS ABAS DE AUDITORIA
# ============================================================

try:
    aba_resumo_pipeline = resumo_pipeline.copy()
except NameError:
    aba_resumo_pipeline = pd.DataFrame()

try:
    aba_resumo_validacoes = resumo_validacoes.copy()
except NameError:
    aba_resumo_validacoes = pd.DataFrame()

try:
    aba_resumo_situacao = resumo_situacao.copy()
except NameError:
    aba_resumo_situacao = pd.DataFrame()

try:
    aba_resumo_idade = resumo_idade.copy()
except NameError:
    aba_resumo_idade = pd.DataFrame()

try:
    aba_auditoria_idade = auditoria_idade_final.copy()
except NameError:
    aba_auditoria_idade = pd.DataFrame()

try:
    aba_auditoria_nomes_possiveis = auditoria_nomes_possiveis.copy()
except NameError:
    aba_auditoria_nomes_possiveis = pd.DataFrame()

try:
    aba_auditoria_nomes_corrigidos = auditoria_nomes_corrigidos.copy()
except NameError:
    aba_auditoria_nomes_corrigidos = pd.DataFrame()

try:
    aba_repetidos_medida = repetidos_medida_protetiva.copy()
except NameError:
    aba_repetidos_medida = pd.DataFrame()

try:
    aba_count_medidas = count_medidas.copy()
except NameError:
    aba_count_medidas = pd.DataFrame()

# Aba resumo consolidada
abas_resumo = []

if len(aba_resumo_pipeline) > 0:
    abas_resumo.append(("Resumo Pipeline", aba_resumo_pipeline))

if len(aba_resumo_validacoes) > 0:
    abas_resumo.append(("Validações", aba_resumo_validacoes))

if len(aba_resumo_situacao) > 0:
    abas_resumo.append(("Resumo Situação", aba_resumo_situacao))

if len(aba_resumo_idade) > 0:
    abas_resumo.append(("Resumo Idade", aba_resumo_idade))

# ============================================================
# 5. EXPORTAR EXCEL COM ABAS
# ============================================================

arquivo_saida = ARQUIVO_SAIDA

with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:
    df_exportar.to_excel(
        writer,
        index=False,
        sheet_name=ABA_PLANILHA_FINAL
    )

    aba_auditoria_idade.to_excel(
        writer,
        index=False,
        sheet_name=ABA_AUDITORIA_IDADE
    )

    aba_auditoria_nomes_possiveis.to_excel(
        writer,
        index=False,
        sheet_name="Possíveis Nomes"
    )

    aba_auditoria_nomes_corrigidos.to_excel(
        writer,
        index=False,
        sheet_name="Nomes Corrigidos"
    )

    aba_repetidos_medida.to_excel(
        writer,
        index=False,
        sheet_name=ABA_REPETIDOS_MEDIDA
    )

    aba_count_medidas.to_excel(
        writer,
        index=False,
        sheet_name="Count Medidas"
    )

    # Escreve os resumos em uma única aba, um abaixo do outro
    startrow = 0

    if len(abas_resumo) == 0:
        pd.DataFrame({"Resumo": ["Nenhum resumo gerado."]}).to_excel(
            writer,
            index=False,
            sheet_name=ABA_RESUMO
        )
    else:
        for titulo, df_resumo in abas_resumo:
            pd.DataFrame({titulo: []}).to_excel(
                writer,
                index=False,
                sheet_name=ABA_RESUMO,
                startrow=startrow
            )

            df_resumo.to_excel(
                writer,
                index=False,
                sheet_name=ABA_RESUMO,
                startrow=startrow + 1
            )

            startrow += len(df_resumo) + 4

# ============================================================
# 6. FORMATAR EXCEL
# ============================================================

wb = load_workbook(arquivo_saida)

header_fill = PatternFill("solid", fgColor="7030A0")
header_font = Font(color="FFFFFF", bold=True)

thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9")
)

for ws in wb.worksheets:
    # Formatar cabeçalho
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )
        cell.border = thin_border

    # Formatar corpo
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )
            cell.border = thin_border

    # Congelar cabeçalho
    ws.freeze_panes = "A2"

    # Aplicar filtro
    if ws.max_row > 1 and ws.max_column > 1:
        ws.auto_filter.ref = ws.dimensions

    # Ajustar largura das colunas
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)

        max_length = 0

        for cell in col:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        largura = min(max(max_length + 2, 12), 45)
        ws.column_dimensions[col_letter].width = largura

# ============================================================
# 7. LISTA SUSPENSA NA COLUNA SITUAÇÃO
# ============================================================

ws_final = wb[ABA_PLANILHA_FINAL]

col_situacao_excel = None

for col in range(1, ws_final.max_column + 1):
    if ws_final.cell(row=1, column=col).value == "Situação":
        col_situacao_excel = col
        break

if col_situacao_excel:
    letra_situacao = get_column_letter(col_situacao_excel)

    valores_situacao = sorted(
        df_exportar["Situação"]
        .dropna()
        .astype(str)
        .unique()
    )

    formula_situacao = '"' + ",".join(valores_situacao) + '"'

    dv = DataValidation(
        type="list",
        formula1=formula_situacao,
        allow_blank=True
    )

    ws_final.add_data_validation(dv)
    dv.add(f"{letra_situacao}2:{letra_situacao}{ws_final.max_row}")

# ============================================================
# 8. SALVAR E BAIXAR
# ============================================================

wb.save(arquivo_saida)

files.download(arquivo_saida)

# ============================================================
# 9. CONFERÊNCIA FINAL
# ============================================================

print("Exportação final concluída com sucesso!")
print(f"Arquivo gerado: {arquivo_saida}")
print(f"Total de linhas exportadas: {len(df_exportar)}")
print(f"Total de colunas exportadas: {len(df_exportar.columns)}")

print("\nAbas criadas:")
for aba in wb.sheetnames:
    print("-", aba)

print("\nPrévia da planilha final:")
display(df_exportar.head(20))